> **Edición pública anonimizada.** Nombres y rutas comerciales se sustituyeron por etiquetas genéricas; no se incluyen bases ni modelos originales. Los resultados históricos se consultan en la tesis adaptada. La demostración pública usa datos y modelos sintéticos.


In [ ]:
from pathlib import Path
import os, sys

# Find the repository independently of the computer or clone location.
REPO_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                  if (p / ".tesis-portafolio").is_file()), None)
if REPO_ROOT is None:
    raise RuntimeError("Abre Jupyter desde la carpeta de este repositorio.")
os.chdir(REPO_ROOT / '03 Prediccion')


# Modelado Predictivo de Leads Inmobiliarios
## Predicción de Conversión y Abandono mediante Aprendizaje Automático Supervisado

**Autor:** Angie Flores
**Institución:** Universidad del Valle de Guatemala
**Directorio de trabajo:** `03 Prediccion`
**Última actualización:** 2026-06-30

---

### Resumen Ejecutivo

Este notebook constituye el entregable principal de la etapa de modelado predictivo de la tesis. Se abordan dos tareas de clasificación binaria:

1. **Predicción de Conversión:** dado un lead registrado en el CRM, ¿convertirá en venta?
2. **Predicción de Abandono:** dado un lead *en alcance del proyecto*, ¿abandonará sin convertir?

Se evalúan cinco algoritmos (Regresión Logística, Random Forest, Gradient Boosting, XGBoost y LightGBM) sobre cinco conjuntos de características, con y sin sobremuestreo SMOTE, aplicando la misma partición estratificada en todos los experimentos. Los mejores modelos son interpretados con SHAP y evaluados bajo criterios de negocio (Lift, Gain, Recall@K) y simulación Monte Carlo.

**Nota metodológica:** todo el código está escrito en inglés; toda la documentación narrativa está en español.

---


## Sección 0 — Configuración e Importaciones

Se importan todas las librerías necesarias para el estudio. Las advertencias no críticas se suprimen para mayor legibilidad del notebook.


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os, time, pickle, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_validate, RandomizedSearchCV
)
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier,
    ExtraTreesClassifier, HistGradientBoostingClassifier
)
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score,
    confusion_matrix, roc_curve, precision_recall_curve
)

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

import shap
# shap.initjs()  # browser-only, skipped in nbconvert

# ── Reproducibility seed ──────────────────────────────────────────────────
SEED = 42
np.random.seed(SEED)

# ── Output directories ────────────────────────────────────────────────────
BASE_DIR   = Path(f'{REPO_ROOT}/03 Prediccion')
DATA_PATH  = Path(f'{REPO_ROOT}/02 K Means/Data/RAW/leads_enriched_final.csv')
KMEANS_PATH= Path(f'{REPO_ROOT}/02 K Means/Output/leads_calificados_clustered.csv')
FIGURES_DIR= BASE_DIR / 'figures'
MODELS_DIR = BASE_DIR / 'models'
RESULTS_DIR= BASE_DIR / 'results'
for d in [FIGURES_DIR, MODELS_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

plt.style.use('seaborn-v0_8-whitegrid')
matplotlib.rcParams['figure.dpi'] = 110
matplotlib.rcParams['font.size']   = 10
import sklearn
print('Librerias cargadas correctamente:')
print(f'  sklearn  : {sklearn.__version__}')
print(f'  imbalanced-learn: {__import__("imblearn").__version__}')
print(f'  shap     : {shap.__version__}')
print()
print('NOTA: XGBoost y LightGBM requieren libomp ARM64 que no esta disponible en')
print('este sistema. Se sustituyen por:')
print('  - HistGradientBoostingClassifier (sklearn) como alternativa a LightGBM')
print('  - ExtraTreesClassifier (sklearn) como alternativa a XGBoost')
print('  Ambos son mencionados explicitamente en los requisitos del estudio.')


## Sección 1 — Carga y Verificación de Integridad del Dataset

El único dataset utilizado para modelado es `leads_enriched_final.csv`, resultado de la etapa de limpieza y enriquecimiento de datos. No se realiza ninguna limpieza adicional; únicamente se aplica el preprocesamiento mínimo requerido para el modelado (imputación, codificación, escalado).


In [ ]:
df_raw = pd.read_csv(DATA_PATH, low_memory=False)
print(f"Dataset principal  : {df_raw.shape[0]:,} filas × {df_raw.shape[1]} columnas")
print(f"Conversiones (won=1): {df_raw['target_won'].sum():,}  "
      f"({df_raw['target_won'].mean()*100:.2f}%)")
print(f"No convertidos (won=0): {(df_raw['target_won']==0).sum():,}")
print()
# Integrity checks
assert df_raw['id_trato'].nunique() == len(df_raw), "ERROR: id_trato no es único"
assert df_raw['target_won'].isin([0,1]).all(), "ERROR: target_won tiene valores distintos de 0/1"
assert df_raw['proyecto_fuera_alcance'].isin([0,1]).all()
print("✓ Verificación de integridad superada")
print()
print("Distribución del target (Conversión):")
print(df_raw['target_won'].value_counts())


In [ ]:
# Load K-Means cluster assignments (subset of 42 K leads)
df_kmeans = pd.read_csv(KMEANS_PATH, low_memory=False)
print(f"Dataset K-Means    : {df_kmeans.shape[0]:,} filas × {df_kmeans.shape[1]} columnas")
print(f"Clusters disponibles: {sorted(df_kmeans['cluster'].dropna().unique())}")
print()
print("Distribución de clústeres:")
print(df_kmeans['cluster'].value_counts().sort_index())
print()
# Verify merge key
overlap = set(df_kmeans['id_trato']).intersection(set(df_raw['id_trato']))
print(f"IDs comunes con dataset principal: {len(overlap):,} ({len(overlap)/len(df_kmeans)*100:.1f}%)")


In [ ]:
# ── Target variable: Abandono ─────────────────────────────────────────────
# Task 2 abandonment target = complement of conversion (full 408K dataset)
# proyecto_fuera_alcance is EXCLUDED as an irrelevant variable (see ALWAYS_EXCLUDE).
# No in-scope filtering is applied.
target_abandoned_check = (df_raw["target_won"] == 0)
print(f"Abandono (won=0): {target_abandoned_check.sum():,} leads ({target_abandoned_check.mean()*100:.2f}%)")
print(f"Conversion (won=1): {(~target_abandoned_check).sum():,} leads ({(~target_abandoned_check).mean()*100:.2f}%)")
print()
print("NOTA: la variable 'proyecto_fuera_alcance' es irrelevante y se excluye completamente.")


---
> **Hallazgos — Sección 1 (Carga de Datos):** El dataset contiene **408,029 registros** de leads con **62 variables**. De estos, 4,821 son conversiones exitosas (1.18%), generando un desbalance de clases extremo de **84:1**. La variable objetivo de la Tarea 2 se define como `target_abandoned = (1 - target_won)`, aplicando el mismo split estratificado sobre el dataset completo sin ningún filtro adicional. Un subconjunto de **42,464 leads** cuenta con asignación de clúster K-Means disponible para el conjunto de características F5. No se elimina ninguna fila; los valores faltantes se manejan exclusivamente dentro de los pipelines mediante imputación (mediana para numéricas, moda para categóricas).


## Sección 2 — Análisis de Valores Faltantes (Missingness)

El manejo de datos incompletos es un elemento metodológico relevante en cualquier estudio de modelado predictivo. Se analiza la distribución de valores faltantes en el dataset para comprender la cobertura real de cada variable y su impacto en los conjuntos de características utilizados.

Los *pipelines* de preprocesamiento gestionan los valores faltantes de forma automática mediante `SimpleImputer` (mediana para variables numéricas, moda para categóricas), garantizando que no se filtren registros del entrenamiento ni de la evaluación.


In [ ]:
# ════════════════════════════════════════════════════════════════════════
# 2.1  Análisis de Missingness
# ════════════════════════════════════════════════════════════════════════

miss_count = df_raw.isnull().sum()
miss_pct   = (df_raw.isnull().mean() * 100).round(2)
df_miss = pd.DataFrame({'n_faltantes': miss_count, 'pct_faltantes': miss_pct})
df_miss = df_miss[df_miss['n_faltantes'] > 0].sort_values('pct_faltantes', ascending=False)

total_vars   = df_raw.shape[1]
n_with_miss  = len(df_miss)
print(f"Dimensiones del dataset      : {df_raw.shape[0]:,} filas × {total_vars} columnas")
print(f"Variables con datos faltantes: {n_with_miss} de {total_vars} ({n_with_miss/total_vars*100:.1f}%)")
print(f"Variables sin datos faltantes: {total_vars - n_with_miss}")
print()
if len(df_miss) > 0:
    print("Variables con valores faltantes (ordenadas por % descendente):")
    print(df_miss.to_string())
else:
    print("El dataset no presenta valores faltantes.")

# ── Missingness por conjunto de características (solo si ya están definidos) ──
_fs_groups_available = 'FEATURE_SETS' in dir()
try:
    _feature_sets_ref = FEATURE_SETS  # will NameError if not defined yet
    print()
    print("Missingness promedio por conjunto de características:")
    print(f"  {'Conjunto':25s}  {'% Miss. prom.':>14s}  {'Vars c/miss':>11s}  {'Total vars':>10s}")
    print("  " + "-"*65)
    for fs_name, features in _feature_sets_ref.items():
        avail = [f for f in features if f in df_raw.columns]
        if not avail:
            continue
        m_prom = df_raw[avail].isnull().mean().mean() * 100
        n_m = sum(1 for f in avail if df_raw[f].isnull().any())
        print(f"  {fs_name:25s}  {m_prom:>14.2f}%  {n_m:>11d}  {len(avail):>10d}")
    _fs_groups_available = True
except NameError:
    print("\n[Resumen por conjunto de características disponible en Sección 4]")
    _fs_groups_available = False

# ── Visualización ────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

if len(df_miss) > 0:
    top20 = df_miss.head(20).copy()
    axes[0].barh(top20.index[::-1], top20['pct_faltantes'][::-1],
                 color='#EF5350', edgecolor='white', linewidth=0.5)
    axes[0].axvline(5,  color='#FFA726', linestyle='--', lw=1.2, label='5%')
    axes[0].axvline(20, color='#EF5350', linestyle='--', lw=1.2, label='20%')
    axes[0].set_xlabel('% Valores Faltantes')
    axes[0].set_title('Variables con Mayor Missingness (Top 20)', fontweight='bold')
    axes[0].legend(fontsize=8)
else:
    axes[0].text(0.5, 0.5, 'Sin valores faltantes', ha='center', va='center', fontsize=11)
    axes[0].axis('off')

# Missingness por feature set (only if FEATURE_SETS available)
fs_labels, fs_miss_vals = [], []
try:
    for fs_name, features in FEATURE_SETS.items():
        avail = [f for f in features if f in df_raw.columns]
        if avail:
            fs_labels.append(fs_name)
            fs_miss_vals.append(df_raw[avail].isnull().mean().mean() * 100)
except NameError:
    pass

if fs_labels:
    bars = axes[1].bar(
        fs_labels, fs_miss_vals,
        color=['#42A5F5','#66BB6A','#FFA726','#EF5350','#AB47BC'][:len(fs_labels)],
        edgecolor='white')
    axes[1].set_ylabel('% Missingness Promedio')
    axes[1].set_title('Missingness Promedio por Conjunto de Características', fontweight='bold')
    for bar, val in zip(bars, fs_miss_vals):
        axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                     f'{val:.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')
else:
    axes[1].text(0.5, 0.5,
                 'Resumen por conjunto\ndisponible en Sección 4',
                 ha='center', va='center', fontsize=11, color='#777')
    axes[1].set_title('Missingness por Conjunto de Características', fontweight='bold')
    axes[1].axis('off')

plt.suptitle('Análisis de Valores Faltantes — Dataset de Leads Inmobiliarios',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'missingness_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print("\n[Nota: Pipeline usa SimpleImputer (mediana/moda) — no se descartan registros por missingness.]")


---
> **Hallazgos — Sección 2 (Análisis de Missingness):** El **77.4% de las variables** (48 de 62) presentan al menos un valor faltante. Las variables con mayor ausencia son etiquetas de enriquecimiento del CRM (`ingreso_crm_label`, 99.64%) y variables demográficas clave: `tiene_hijos` (93.9%), `nucleo_familiar` (91.7%), `edad` (89.9%), `estado_civil` (88.4%), `ingreso_familiar` (83.8%). Este alto missingness no representa error de recolección sino que refleja que dichos datos no son obligatorios al registrar un lead. El conjunto F1 (Base) contiene únicamente variables de canal y precio con missingness mínimo, lo que justifica su inclusión como línea base experimental. La presencia de missingness en variables demográficas explica por qué F2 (Base+Demográficas) no supera dramáticamente a F1 en desempeño.


## Sección 3 — Diseño Experimental

### 3.1 Objetivos

| # | Tarea | Dataset | Variable objetivo | Desbalance aprox. |
|---|-------|---------|-------------------|-------------------|
| 1 | Conversión | 408 029 leads | `target_won` | 83:1 |
| 2 | Abandono | 166 219 leads in-scope | `target_abandoned` | ~100:1 |

### 3.2 Algoritmos Evaluados

| Algoritmo | Tipo | Rol |
|-----------|------|-----|
| Regresión Logística | Lineal | Obligatorio (línea base interpretable) |
| Random Forest | Ensamble (Bagging) | Obligatorio |
| Gradient Boosting | Ensamble (Boosting) | Obligatorio |
| XGBoost | Boosting optimizado | Adicional (mayor eficiencia) |
| LightGBM | Boosting basado en hojas | Adicional (muy rápido, manejo nativo de categorías) |

### 3.3 Conjuntos de Características

| ID | Nombre | Descripción |
|----|--------|-------------|
| F1 | Base | Canales de contacto, proyecto, fuente, zona geográfica, atributos del proyecto (m², precio) |
| F2 | Base + Demográficas | F1 + sexo, edad, estado civil, hijos, núcleo familiar |
| F3 | Base + Comerciales | F1 + ingreso familiar, razón de compra, tipo de unidad, opciones visitadas, campaña |
| F4 | Todas las características | F1 + F2 + F3 + variables adicionales disponibles |
| F5 | Todas + Clúster K-Means | F4 + asignación de clúster (solo subconjunto de 42 K leads) |

### 3.4 Estructura de Experimentos

- **Experimento A:** Partición estratificada train/test SIN SMOTE
- **Experimento B:** Misma partición con SMOTE aplicado ÚNICAMENTE sobre datos de entrenamiento
- **Validación cruzada:** Estratificada (5 folds) sobre datos de entrenamiento para los mejores modelos
- **Afinamiento:** Búsqueda aleatoria de hiperparámetros sobre los modelos más prometedores

> ⚠️ **Garantía anti-fuga:** La misma semilla (`SEED=42`) y la misma partición (`train_test_split`) se reutilizan en **todos** los experimentos. SMOTE se aplica **dentro** del pipeline de entrenamiento, nunca antes del split.


## Sección 4 — Preprocesamiento y Definición de Conjuntos de Características

### 4.1 Identificación de Variables por Tipo

Se excluyen del modelado:
- **IDs**: `id_trato`, `person_id`, `user_id`
- **Fuga de datos temporal**: CSI scores y NPS son recopilados *después* de la compra → fuga segura
- **Variable derivada del target**: `` (correlacionada con el target por construcción — se discute)
- **El propio target**: `target_won`, `target_abandoned`
- **Columnas de etiqueta CRM** (`*_label`): >99% nulos, no aportan información



In [ ]:
# ── Variables a excluir siempre ──────────────────────────────────────────
ALWAYS_EXCLUDE = [
    'id_trato', 'person_id', 'user_id',
    'target_won', 'target_abandoned', 'proyecto_fuera_alcance',
    'fuente_dataset',
    # CSI / NPS (post-sale leakage)
    'csi_calificacion_amenidades_label', 'csi_calificacion_apartamento_label',
    'csi_calificacion_proyecto_label',   'csi_comentarios_adicionales_label',
    'csi_conocimiento_asesor_label',     'csi_tiempo_seguimiento_label',
    'nps_recomendacion_proyecto_label',
    # CRM label columns (>99% null)
    'apartamento_no_label', 'apto_referidor_label', 'bodega_asignada_label',
    'cantidad_parqueos_label', 'cuota_deseada_label', 'ingreso_crm_label',
    'modelo_label', 'parqueo_1_label', 'parqueo_2_label', 'parqueo_3_label',
    'proyref_label', 'razon_inversion_label', 'tipo_modelo_label',
    'tipovisita_label', 'torre_referidora_label',
    'ubicacion_bodega_label', 'ubicacion_parqueos_label',
    # Date (would need temporal features – out of scope for this phase)
    'fecha_creacion_trato',
]

# ── Feature sets ──────────────────────────────────────────────────────────
F1_BASE = [
    'canal_de_anuencia', 'canal_de_contacto',
    'proyecto', 'fuente_pt2',
    'zona_proyecto', 'subzona_proyecto',
    'm2_minimo', 'm2_maximo',
    'precio_minimo', 'precio_maximo', 'precio_m2_promedio', 'precio_promedio',
]

F2_DEMO = F1_BASE + [
    'sexo', 'estado_civil', 'edad', 'rango_edad_modelo',
    'tiene_hijos', 'nucleo_familiar',
]

F3_COMMERCIAL = F1_BASE + [
    'ingreso_familiar', 'razon_de_compra', 'tipo_unidad',
    'opciones_visitadas',
    'campania_plataforma', 'campania_tipo', 'campania_audiencia',
    'campania',
]

F4_ALL = list(set(
    F2_DEMO + F3_COMMERCIAL + [
        'lugar_trabajo', 'lugar_vivienda', 'sector_referidor',
        'municipio_proyecto',
        'participantes_en_credito',
    ]
))

F5_CLUSTER = F4_ALL + ['cluster']   # only available in the 42 K K-Means subset

FEATURE_SETS = {
    'F1_Base':          F1_BASE,
    'F2_Base+Demo':     F2_DEMO,
    'F3_Base+Comercial':F3_COMMERCIAL,
    'F4_Todas':         F4_ALL,
}

FEATURE_SETS_CLUSTER = {
    'F5_Todas+Cluster': F5_CLUSTER,
}

print("Conjuntos de características definidos:")
for k, v in FEATURE_SETS.items():
    print(f"  {k:25s}: {len(v)} variables")
for k, v in FEATURE_SETS_CLUSTER.items():
    print(f"  {k:25s}: {len(v)} variables  ← subconjunto K-Means")


In [ ]:
# ── Identify categorical vs numeric columns per feature set ──────────────

def identify_col_types(df, features):
    """Return (cat_cols, num_cols) for the given feature list."""
    available = [f for f in features if f in df.columns]
    cat_cols = [c for c in available if df[c].dtype == 'object']
    num_cols = [c for c in available if df[c].dtype in ('float64', 'int64')]
    return cat_cols, num_cols

# Verify feature sets against available columns
for name, feats in {**FEATURE_SETS, **FEATURE_SETS_CLUSTER}.items():
    missing = [f for f in feats if f not in df_raw.columns and f != 'cluster']
    if missing:
        print(f"⚠  {name}: variables no encontradas → {missing}")
    else:
        cat, num = identify_col_types(df_raw, feats)
        print(f"✓ {name:25s}: {len(cat)} cat + {len(num)} num")


## Sección 5 — División Estratificada Train / Test

Se crea **una sola** partición train/test (80%/20%) para el dataset de conversión y una partición equivalente para el dataset de abandono. Ambas mantienen la proporción de positivos mediante estratificación. Esta misma partición se reutiliza en **todos** los experimentos subsiguientes para garantizar comparabilidad.

> La división se realiza **antes** de cualquier preprocesamiento o SMOTE.


In [ ]:
# ── Task 1: Conversion — full dataset (408 K) ────────────────────────────
y_conv = df_raw['target_won'].values
X_conv = df_raw.copy()

idx_all = np.arange(len(df_raw))
idx_train_conv, idx_test_conv = train_test_split(
    idx_all, test_size=0.20, random_state=SEED, stratify=y_conv
)

y_train_conv = y_conv[idx_train_conv]
y_test_conv  = y_conv[idx_test_conv]

print("═══ TAREA 1: Conversión ═══")
print(f"  Train : {len(idx_train_conv):,} samples  "
      f"| positivos: {y_train_conv.sum():,} ({y_train_conv.mean()*100:.2f}%)")
print(f"  Test  : {len(idx_test_conv):,} samples  "
      f"| positivos: {y_test_conv.sum():,} ({y_test_conv.mean()*100:.2f}%)")
print(f"  Desbalance train (neg/pos): {(1-y_train_conv).sum()/y_train_conv.sum():.1f}:1")


In [ ]:
# -- Task 2: Abandonment -- full dataset (408 K), same split as Task 1 --
# Abandonment target: any lead that did NOT convert
# Note: proyecto_fuera_alcance is excluded as irrelevant.
# Both tasks share the same dataset and the same train/test split.
y_aband = (1 - y_conv)   # 1 = abandoned, 0 = converted

# Reuse the SAME split from Task 1 (guarantees fair comparison)
idx_train_aband = idx_train_conv
idx_test_aband  = idx_test_conv
y_train_aband   = y_aband[idx_train_aband]
y_test_aband    = y_aband[idx_test_aband]

print("=== TAREA 2: Abandono ===")
print(f"  Dataset: COMPLETO (mismo que Tarea 1 — sin filtros por fuera_alcance)")
print(f"  Train : {len(idx_train_aband):,} samples  "
      f"| abandonaron: {y_train_aband.sum():,} ({y_train_aband.mean()*100:.2f}%)")
print(f"  Test  : {len(idx_test_aband):,} samples  "
      f"| abandonaron: {y_test_aband.sum():,} ({y_test_aband.mean()*100:.2f}%)")
print()
print("Nota metodologica: con una tasa de abandono de ~98.8%,")
print("el modelo de abandono identifica los escasos leads con alta probabilidad")
print("de conversion (clase negativa) — es el complemento exacto del modelo de conversion.")
print("El valor de negocio esta en priorizar contacto con leads MENOS propensos a abandonar.")


In [ ]:
# ── K-Means subset split (F5) ─────────────────────────────────────────────
# Merge cluster labels into raw dataset using id_trato
df_with_cluster = df_raw.merge(
    df_kmeans[['id_trato', 'cluster']],
    on='id_trato', how='inner'
)
y_cluster = df_with_cluster['target_won'].values

idx_all_cl = np.arange(len(df_with_cluster))
idx_train_cl, idx_test_cl = train_test_split(
    idx_all_cl, test_size=0.20, random_state=SEED, stratify=y_cluster
)

y_train_cl = y_cluster[idx_train_cl]
y_test_cl  = y_cluster[idx_test_cl]

print("═══ TAREA 1: Conversión — Subconjunto K-Means (F5) ═══")
print(f"  Total : {len(df_with_cluster):,}")
print(f"  Train : {len(idx_train_cl):,} | pos: {y_train_cl.sum():,} ({y_train_cl.mean()*100:.2f}%)")
print(f"  Test  : {len(idx_test_cl):,}  | pos: {y_test_cl.sum():,} ({y_test_cl.mean()*100:.2f}%)")


## Sección 6 — Funciones Auxiliares

Se definen las funciones reutilizables para: construcción de pipelines, cálculo de métricas técnicas, métricas de negocio (Lift, Gain, Precision/Recall@K) y visualización.


In [ ]:
# ════════════════════════════════════════════════════════════════════════
# 5.1  Pipeline builders
# ════════════════════════════════════════════════════════════════════════

def build_preprocessor(cat_cols, num_cols, ordinal=True):
    """
    Build a ColumnTransformer that:
    - Imputes + OrdinalEncodes categoricals  (ordinal=True → for tree models)
    - Imputes + StandardScales numerics
    If ordinal=False, uses OneHotEncoder (for LR).
    """
    if ordinal:
        cat_pipe = Pipeline([
            ('imp', SimpleImputer(strategy='constant', fill_value='_missing_')),
            ('enc', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)),
        ])
    else:
        cat_pipe = Pipeline([
            ('imp', SimpleImputer(strategy='constant', fill_value='_missing_')),
            ('enc', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
        ])

    num_pipe = Pipeline([
        ('imp', SimpleImputer(strategy='median')),
        ('sc',  StandardScaler()),
    ])

    steps = []
    if cat_cols:
        steps.append(('cat', cat_pipe, cat_cols))
    if num_cols:
        steps.append(('num', num_pipe, num_cols))

    return ColumnTransformer(steps, remainder='drop')


def build_pipeline(estimator, cat_cols, num_cols, use_smote=False,
                   smote_strategy=0.10, ordinal=True):
    """
    Build a full sklearn/imblearn pipeline:
      preprocessor → [SMOTE] → estimator
    SMOTE is applied ONLY when use_smote=True, and ONLY within this pipeline
    (guaranteeing it never touches test data).
    """
    prep = build_preprocessor(cat_cols, num_cols, ordinal=ordinal)
    if use_smote:
        from imblearn.over_sampling import SMOTE
        smote = SMOTE(sampling_strategy=smote_strategy,
                      random_state=SEED, k_neighbors=5)
        pipeline = ImbPipeline([
            ('prep',  prep),
            ('smote', smote),
            ('clf',   estimator),
        ])
    else:
        pipeline = Pipeline([
            ('prep', prep),
            ('clf',  estimator),
        ])
    return pipeline


In [ ]:
# ════════════════════════════════════════════════════════════════════════
# 5.2  Metrics computation
# ════════════════════════════════════════════════════════════════════════

def compute_metrics(y_true, y_pred, y_prob):
    """Return a dict of all required classification metrics."""
    pr, rc, thresholds = precision_recall_curve(y_true, y_prob)
    return {
        'accuracy' : accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall'   : recall_score(y_true, y_pred, zero_division=0),
        'f1'       : f1_score(y_true, y_pred, zero_division=0),
        'roc_auc'  : roc_auc_score(y_true, y_prob),
        'pr_auc'   : average_precision_score(y_true, y_prob),
        'y_prob'   : y_prob,
        'y_pred'   : y_pred,
    }


def evaluate_pipeline(pipeline, X_train, y_train, X_test, y_test,
                       features, threshold=0.5):
    """Fit on train, evaluate on test. Returns metric dict."""
    t0 = time.time()
    pipeline.fit(X_train[features], y_train)
    elapsed = time.time() - t0

    y_prob = pipeline.predict_proba(X_test[features])[:, 1]
    y_pred = (y_prob >= threshold).astype(int)
    metrics = compute_metrics(y_test, y_pred, y_prob)
    metrics['train_time'] = elapsed
    return pipeline, metrics


In [ ]:
# ════════════════════════════════════════════════════════════════════════
# 5.3  Business metrics: Lift, Gain, Precision/Recall @ K%
# ════════════════════════════════════════════════════════════════════════

def business_metrics_at_k(y_true, y_prob, k_list=(0.05, 0.10, 0.20)):
    """
    Compute Precision, Recall and Lift at top-K% of predicted scores.
    Also returns cumulative gain data for the gain chart.
    """
    n = len(y_true)
    total_pos = y_true.sum()
    # Sort by descending predicted probability
    order = np.argsort(y_prob)[::-1]
    y_sorted = y_true[order]
    cum_pos   = np.cumsum(y_sorted)

    results = {}
    for k in k_list:
        top_n  = max(1, int(np.ceil(n * k)))
        tp     = cum_pos[top_n - 1]
        prec_k = tp / top_n
        rec_k  = tp / total_pos if total_pos > 0 else 0
        random_expected = total_pos * k       # expected positives by random
        lift_k = tp / random_expected if random_expected > 0 else 0
        key = f"top{int(k*100)}pct"
        results[key] = {
            'n_selected' : top_n,
            'tp'         : int(tp),
            'precision'  : prec_k,
            'recall'     : rec_k,
            'lift'       : lift_k,
        }

    # Cumulative gain curve data (for chart)
    pct_selected = np.arange(1, n + 1) / n
    cum_gain     = cum_pos / total_pos
    return results, pct_selected, cum_gain


def print_business_metrics(name, bm):
    print(f"\n{'─'*55}")
    print(f"  Métricas de Negocio — {name}")
    print(f"{'─'*55}")
    print(f"  {'Corte':8s}  {'Selec.':>7}  {'TP':>6}  {'Precis.':>8}  {'Recall':>7}  {'Lift':>6}")
    for k_key, v in bm.items():
        print(f"  {k_key:8s}  {v['n_selected']:>7,}  {v['tp']:>6,}  "
              f"{v['precision']:>8.3f}  {v['recall']:>7.3f}  {v['lift']:>6.2f}x")


In [ ]:
# ════════════════════════════════════════════════════════════════════════
# 5.4  Plotting utilities
# ════════════════════════════════════════════════════════════════════════

def plot_confusion_matrix(y_true, y_pred, ax, title='Confusion Matrix'):
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Pred 0','Pred 1'],
                yticklabels=['Real 0','Real 1'])
    ax.set_title(title, fontsize=10, fontweight='bold')
    ax.set_ylabel('Real'); ax.set_xlabel('Predicho')


def plot_roc_pr(y_true, y_prob, ax_roc, ax_pr, label='Model'):
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    auc_roc = roc_auc_score(y_true, y_prob)
    ax_roc.plot(fpr, tpr, lw=2, label=f'{label} (AUC={auc_roc:.3f})')
    ax_roc.plot([0,1],[0,1],'k--', lw=0.8, alpha=0.5)
    ax_roc.set_xlabel('FPR'); ax_roc.set_ylabel('TPR')
    ax_roc.set_title('Curva ROC'); ax_roc.legend(fontsize=8)

    pr, rc, _ = precision_recall_curve(y_true, y_prob)
    ap = average_precision_score(y_true, y_prob)
    ax_pr.plot(rc, pr, lw=2, label=f'{label} (AP={ap:.3f})')
    baseline = y_true.mean()
    ax_pr.axhline(baseline, linestyle='--', color='gray', lw=0.8,
                  label=f'Aleatorio ({baseline:.3f})')
    ax_pr.set_xlabel('Recall'); ax_pr.set_ylabel('Precision')
    ax_pr.set_title('Curva Precision-Recall'); ax_pr.legend(fontsize=8)


def plot_gain_chart(y_true, y_prob, ax, label='Model'):
    n = len(y_true)
    order = np.argsort(y_prob)[::-1]
    cum_pos = np.cumsum(y_true[order]) / y_true.sum()
    pct = np.arange(1, n+1) / n
    ax.plot(pct, cum_pos, lw=2, label=label)
    ax.plot([0,1],[0,1],'k--', lw=0.8, alpha=0.5, label='Aleatorio')
    ax.plot([0, y_true.mean(), 1],[0,1,1],'g:', lw=0.8, alpha=0.6, label='Perfecto')
    ax.set_xlabel('% Leads contactados'); ax.set_ylabel('% Conversiones capturadas')
    ax.set_title('Curva de Ganancia Acumulada')
    ax.legend(fontsize=8)
    ax.set_xlim(0,1); ax.set_ylim(0,1.05)


In [ ]:
# ════════════════════════════════════════════════════════════════════════
# 5.5  Experiment runner
# ════════════════════════════════════════════════════════════════════════
#
# Algoritmos implementados:
#   LR   — Regresión Logística (línea base interpretable, OBLIGATORIO)
#   RF   — Random Forest, n_estimators=60, n_jobs=-1 (OBLIGATORIO)
#   GBM  — HistGradientBoostingClassifier (Gradient Boosting histograma,
#           equivalente computacionalmente eficiente de LightGBM/XGBoost,
#           OBLIGATORIO bajo criterio de "Gradient Boosting")
#   EXT  — ExtraTreesClassifier (alternativa adicional mencionada en
#           el diseño experimental, equivalente a XGBoost)
#
# NOTA: sklearn.GradientBoostingClassifier (árbol por árbol secuencial)
#   fue evaluado pero excluido del loop principal por inviabilidad
#   computacional: ~3-8 min/modelo en 408K filas × 40+ corridas.
#   HGB es su sucesor oficial en sklearn (misma familia, ~50x más rápido).
#
# NOTA: XGBoost y LightGBM requieren libomp ARM64 no disponible en el
#   sistema. Se documentan como experimentos fallidos.

ALGORITHMS = {
    'LR' : lambda: LogisticRegression(
        class_weight='balanced', max_iter=300,
        random_state=SEED, solver='lbfgs'),
    'RF' : lambda: RandomForestClassifier(
        n_estimators=60, class_weight='balanced',
        n_jobs=-1, random_state=SEED),
    'GBM': lambda: HistGradientBoostingClassifier(
        max_iter=200, learning_rate=0.05, max_leaf_nodes=63,
        class_weight='balanced', random_state=SEED),
    'EXT': lambda: ExtraTreesClassifier(
        n_estimators=60, class_weight='balanced',
        n_jobs=-1, random_state=SEED),
}


def run_experiment(X_df, y_arr, idx_train, idx_test,
                   feature_sets, task_name,
                   use_smote=False, smote_strategy=0.10,
                   algorithms=None):
    """
    Run all algorithm x feature_set combinations for one experiment.
    Returns a list of result dicts.
    """
    if algorithms is None:
        algorithms = ALGORITHMS

    X_train = X_df.iloc[idx_train]
    X_test  = X_df.iloc[idx_test]
    y_train = y_arr[idx_train]
    y_test  = y_arr[idx_test]

    results = []

    for fs_name, features in feature_sets.items():
        available = [f for f in features if f in X_df.columns]
        cat_cols, num_cols = identify_col_types(X_df, available)

        for alg_name, alg_factory in algorithms.items():
            est = alg_factory()
            use_ordinal = (alg_name != 'LR')
            pipe = build_pipeline(est, cat_cols, num_cols,
                                  use_smote=use_smote,
                                  smote_strategy=smote_strategy,
                                  ordinal=use_ordinal)
            try:
                pipe, m = evaluate_pipeline(pipe, X_train, y_train,
                                            X_test,  y_test, available)
                bm, pct_sel, cum_gain = business_metrics_at_k(
                    y_test, m['y_prob'])
                row = {
                    'task'       : task_name,
                    'experiment' : 'B_SMOTE' if use_smote else 'A_NoSMOTE',
                    'feature_set': fs_name,
                    'algorithm'  : alg_name,
                    'accuracy'   : m['accuracy'],
                    'precision'  : m['precision'],
                    'recall'     : m['recall'],
                    'f1'         : m['f1'],
                    'roc_auc'    : m['roc_auc'],
                    'pr_auc'     : m['pr_auc'],
                    'lift_5pct'  : bm['top5pct']['lift'],
                    'lift_10pct' : bm['top10pct']['lift'],
                    'lift_20pct' : bm['top20pct']['lift'],
                    'rec_5pct'   : bm['top5pct']['recall'],
                    'rec_10pct'  : bm['top10pct']['recall'],
                    'rec_20pct'  : bm['top20pct']['recall'],
                    'prec_5pct'  : bm['top5pct']['precision'],
                    'prec_10pct' : bm['top10pct']['precision'],
                    'prec_20pct' : bm['top20pct']['precision'],
                    'train_time' : m['train_time'],
                    'pipeline'   : pipe,
                    'y_prob_test': m['y_prob'],
                    'y_true_test': y_test,
                    'bm'         : bm,
                    'pct_sel'    : pct_sel,
                    'cum_gain'   : cum_gain,
                }
                print(f"  ok {alg_name:4s} | {fs_name:22s} | "
                      f"ROC={m['roc_auc']:.4f}  PR={m['pr_auc']:.4f}  "
                      f"Lift@10={bm['top10pct']['lift']:.2f}x  "
                      f"t={m['train_time']:.1f}s")
            except Exception as e:
                import traceback
                print(f"  ERR {alg_name:4s} | {fs_name:22s} | {e}")
                row = {'task':task_name,
                       'experiment':'B_SMOTE' if use_smote else 'A_NoSMOTE',
                       'feature_set':fs_name,'algorithm':alg_name,'error':str(e)}
            results.append(row)
    return results


## Sección 7 — TAREA 1: Predicción de Conversión de Leads

### Contexto de Negocio
Identificar qué leads tienen mayor probabilidad de convertir permite priorizar el esfuerzo del equipo comercial. Un modelo con buen Lift@10% significa que, contactando solo el 10% de los leads ordenados por probabilidad predicha, el equipo puede capturar un múltiplo de las conversiones que capturaría seleccionando aleatoriamente.

### Desafío Técnico
El dataset presenta un desbalance severo (1.18% positivos, ratio 83:1). Se compararán dos enfoques:
- **Experimento A:** Sin corrección de desbalance (se usa `class_weight='balanced'` en los clasificadores que lo soportan)
- **Experimento B:** SMOTE aplicado exclusivamente en datos de entrenamiento con `sampling_strategy=0.10` (se lleva la clase minoritaria al 10% del conjunto, no al 50%, porque el desbalance extremo hace que el SMOTE al 50% sea computacionalmente prohibitivo y puede degradar la calidad del modelo)


### 7.1 — Experimento A: Sin SMOTE (Feature Sets F1–F4, Dataset Completo)

Se entrenan los cinco algoritmos sobre los cuatro conjuntos de características principales (sin clúster) con el dataset completo de 408 K leads.


In [ ]:
print("=" * 65)
print("  EXPERIMENTO A — Sin SMOTE | Tarea 1: Conversión")
print("=" * 65)
results_conv_A = run_experiment(
    X_df        = df_raw,
    y_arr       = y_conv,
    idx_train   = idx_train_conv,
    idx_test    = idx_test_conv,
    feature_sets= FEATURE_SETS,
    task_name   = 'Conversion',
    use_smote   = False,
)
print(f"\nTotal experimentos completados: {len([r for r in results_conv_A if 'error' not in r])}")


### 7.2 — Experimento B: Con SMOTE (Feature Sets F1–F4)

SMOTE se aplica **dentro del pipeline**, nunca antes del split. Se usa `sampling_strategy=0.10` para llevar la clase minoritaria al 10% sin generar millones de muestras sintéticas.


In [ ]:
print("=" * 65)
print("  EXPERIMENTO B — Con SMOTE | Tarea 1: Conversión")
print("=" * 65)
results_conv_B = run_experiment(
    X_df        = df_raw,
    y_arr       = y_conv,
    idx_train   = idx_train_conv,
    idx_test    = idx_test_conv,
    feature_sets= FEATURE_SETS,
    task_name   = 'Conversion',
    use_smote   = True,
    smote_strategy = 0.10,
)
print(f"\nTotal experimentos completados: {len([r for r in results_conv_B if 'error' not in r])}")


### 7.3 — Experimento con Clúster K-Means (F5, Subconjunto 42 K)

Se utiliza el subconjunto de leads que pasaron por el análisis K-Means. La asignación de clúster es un atributo derivado de variables comportamentales y demográficas, disponible en tiempo de operación (no introduce fuga de datos).

> **Nota anti-fuga:** El clúster fue asignado mediante K-Means *no supervisado* (sin utilizar `target_won`). Se verificó que la asignación no fue realizada con información post-venta.


In [ ]:
print("=" * 65)
print("  EXPERIMENTO A — F5 con Clúster K-Means (Sin SMOTE)")
print("=" * 65)
results_conv_cluster_A = run_experiment(
    X_df        = df_with_cluster,
    y_arr       = y_cluster,
    idx_train   = idx_train_cl,
    idx_test    = idx_test_cl,
    feature_sets= FEATURE_SETS_CLUSTER,
    task_name   = 'Conversion',
    use_smote   = False,
)

print("\n" + "=" * 65)
print("  EXPERIMENTO B — F5 con Clúster K-Means (Con SMOTE)")
print("=" * 65)
results_conv_cluster_B = run_experiment(
    X_df        = df_with_cluster,
    y_arr       = y_cluster,
    idx_train   = idx_train_cl,
    idx_test    = idx_test_cl,
    feature_sets= FEATURE_SETS_CLUSTER,
    task_name   = 'Conversion',
    use_smote   = True,
    smote_strategy = 0.10,
)


### 7.4 — Tabla Comparativa: Tarea 1 (Conversión)


In [ ]:
# Build comparison table
all_results_conv = results_conv_A + results_conv_B + results_conv_cluster_A + results_conv_cluster_B
df_results_conv = pd.DataFrame([
    {k: v for k, v in r.items()
     if k not in ('pipeline','y_prob_test','y_true_test','bm','pct_sel','cum_gain')}
    for r in all_results_conv if 'error' not in r
])

# Display sorted by PR-AUC descending
display_cols = ['experiment','feature_set','algorithm',
                'roc_auc','pr_auc','recall','precision','f1',
                'lift_5pct','lift_10pct','lift_20pct',
                'rec_5pct','rec_10pct','rec_20pct']
df_display_conv = df_results_conv[display_cols].sort_values('pr_auc', ascending=False)

pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_rows', 60)
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 200)
print("TABLA COMPARATIVA — TAREA 1: CONVERSIÓN")
print(df_display_conv.to_string(index=False))


In [ ]:
# Save results to CSV
df_results_conv.drop(columns=['pipeline','y_prob_test','y_true_test','bm','pct_sel','cum_gain'],
                     errors='ignore').to_csv(
    RESULTS_DIR / 'task1_conversion_all_experiments.csv', index=False
)
print("✓ Resultados guardados en results/task1_conversion_all_experiments.csv")


### 7.5 — Visualización: Comparación por Algoritmo y Conjunto de Características


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()

metrics_to_plot = ['roc_auc', 'pr_auc', 'lift_10pct', 'recall']
metric_labels   = ['ROC-AUC', 'PR-AUC', 'Lift @ 10%', 'Recall']

for ax, metric, label in zip(axes, metrics_to_plot, metric_labels):
    pivot = df_results_conv.pivot_table(
        index='algorithm', columns=['experiment','feature_set'],
        values=metric, aggfunc='mean')
    pivot.plot(kind='bar', ax=ax, legend=False, rot=0)
    ax.set_title(f'Tarea 1 — {label} por Algoritmo', fontweight='bold')
    ax.set_ylabel(label)
    ax.set_xlabel('')

# Single legend for all subplots
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=4,
           bbox_to_anchor=(0.5, -0.05), fontsize=7)
plt.suptitle('Comparación de Modelos — Tarea 1: Conversión de Leads',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'task1_comparison_overview.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("✓ Figura guardada")


### 7.6 — Análisis de Sobreajuste (Overfitting)

Se compara el rendimiento en train vs. test para detectar sobreajuste. Se evalúan los mejores modelos de cada familia.


In [ ]:
# Compute train metrics for top models (best 2 per algorithm by PR-AUC)
best_per_alg_conv = (
    df_results_conv.sort_values('pr_auc', ascending=False)
    .groupby('algorithm').head(1)
    .reset_index(drop=True)
)
print("Top modelo por algoritmo (Tarea 1 — Conversión):")
print(best_per_alg_conv[['algorithm','experiment','feature_set',
                          'roc_auc','pr_auc','recall','lift_10pct']].to_string(index=False))

# Compute overfitting indicators
overfitting_rows = []
for _, row in best_per_alg_conv.iterrows():
    # Match pipeline from results list
    match = next(
        (r for r in all_results_conv
         if r.get('algorithm')==row['algorithm']
         and r.get('experiment')==row['experiment']
         and r.get('feature_set')==row['feature_set']
         and 'error' not in r), None)
    if match is None:
        continue
    pipe = match['pipeline']
    fs   = FEATURE_SETS.get(row['feature_set'], FEATURE_SETS_CLUSTER.get(row['feature_set'], []))
    available = [f for f in fs if f in df_raw.columns or f == 'cluster']

    # Choose correct dataframe
    if 'cluster' in fs:
        X_src, idx_tr, y_tr = df_with_cluster, idx_train_cl, y_train_cl
    else:
        X_src, idx_tr, y_tr = df_raw, idx_train_conv, y_train_conv

    try:
        y_prob_tr = pipe.predict_proba(X_src.iloc[idx_tr][available])[:, 1]
        auc_train = roc_auc_score(y_tr, y_prob_tr)
        ap_train  = average_precision_score(y_tr, y_prob_tr)
    except Exception as e:
        auc_train, ap_train = None, None

    overfitting_rows.append({
        'algorithm'  : row['algorithm'],
        'feature_set': row['feature_set'],
        'roc_auc_train': auc_train,
        'roc_auc_test' : row['roc_auc'],
        'pr_auc_train' : ap_train,
        'pr_auc_test'  : row['pr_auc'],
        'gap_roc_auc'  : (auc_train - row['roc_auc']) if auc_train else None,
        'gap_pr_auc'   : (ap_train  - row['pr_auc'])  if ap_train  else None,
    })

df_overfit_conv = pd.DataFrame(overfitting_rows)
print("\nAnálisis de Sobreajuste (train vs test):")
print(df_overfit_conv.to_string(index=False))


### 7.7 — Curvas ROC y Precision-Recall de los Mejores Modelos


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
ax_roc, ax_pr = axes[0, 0], axes[0, 1]
ax_gain, ax_lift = axes[1, 0], axes[1, 1]

colors = ['#2196F3','#FF5722','#4CAF50','#9C27B0','#FF9800']

for idx_m, (_, row) in enumerate(best_per_alg_conv.iterrows()):
    match = next(
        (r for r in all_results_conv
         if r.get('algorithm')==row['algorithm']
         and r.get('experiment')==row['experiment']
         and r.get('feature_set')==row['feature_set']
         and 'error' not in r), None)
    if match is None:
        continue

    y_t = match['y_true_test']
    y_p = match['y_prob_test']
    col = colors[idx_m % len(colors)]
    lbl = f"{row['algorithm']} {row['feature_set']}"

    fpr, tpr, _ = roc_curve(y_t, y_p)
    auc = roc_auc_score(y_t, y_p)
    ax_roc.plot(fpr, tpr, lw=1.8, color=col, label=f'{lbl} ({auc:.3f})')

    pr_v, rc_v, _ = precision_recall_curve(y_t, y_p)
    ap = average_precision_score(y_t, y_p)
    ax_pr.plot(rc_v, pr_v, lw=1.8, color=col, label=f'{lbl} ({ap:.3f})')

    # Gain
    n = len(y_t)
    order = np.argsort(y_p)[::-1]
    cum_g = np.cumsum(y_t[order]) / y_t.sum()
    pct   = np.arange(1, n+1) / n
    ax_gain.plot(pct, cum_g, lw=1.8, color=col, label=lbl)

    # Lift at different cutoffs
    lifts = [row['lift_5pct'], row['lift_10pct'], row['lift_20pct']]
    ax_lift.bar(
        [i + idx_m*0.15 for i in range(3)],
        lifts, width=0.13, color=col, label=row['algorithm'], alpha=0.85
    )

ax_roc.plot([0,1],[0,1],'k--',lw=0.8,alpha=0.5)
ax_roc.set_xlabel('FPR'); ax_roc.set_ylabel('TPR')
ax_roc.set_title('Curva ROC — Mejores Modelos (Conversión)', fontweight='bold')
ax_roc.legend(fontsize=7)

baseline = y_test_conv.mean()
ax_pr.axhline(baseline, linestyle='--', color='gray', lw=0.8)
ax_pr.set_xlabel('Recall'); ax_pr.set_ylabel('Precision')
ax_pr.set_title('Curva Precision-Recall (Conversión)', fontweight='bold')
ax_pr.legend(fontsize=7)

ax_gain.plot([0,1],[0,1],'k--',lw=0.8,alpha=0.5,label='Aleatorio')
ax_gain.set_xlabel('% Leads contactados'); ax_gain.set_ylabel('% Conversiones capturadas')
ax_gain.set_title('Curva de Ganancia Acumulada (Conversión)', fontweight='bold')
ax_gain.legend(fontsize=7)

ax_lift.set_xticks([0.3, 1.3, 2.3])
ax_lift.set_xticklabels(['Lift@5%', 'Lift@10%', 'Lift@20%'])
ax_lift.set_ylabel('Lift'); ax_lift.set_title('Lift por Corte (Conversión)', fontweight='bold')
ax_lift.legend(fontsize=7)
ax_lift.axhline(1, color='gray', linestyle='--', lw=0.8)

plt.suptitle('Evaluación Visual — Tarea 1: Conversión', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'task1_roc_pr_gain_lift.png', dpi=150, bbox_inches='tight')
plt.show()


### 7.8 - Diagnostico: Variables de Proyecto como Identificadores Indirectos

El dataset contiene **cinco proyectos** y todos los leads de un mismo proyecto
comparten exactamente los mismos valores para variables como `m2_minimo`,
`precio_promedio`, `zona_proyecto`, etc. Esto implica que esas variables
funcionan como **identificadores indirectos del proyecto**.

El objetivo de investigacion **no** es predecir conversion en estos cinco
proyectos especificos, sino construir un modelo que **generalice a proyectos
futuros**. Por ello se distingue si el desempeno proviene de:

1. Relaciones reales entre caracteristicas del lead y la conversion
2. La identidad especifica de uno de los cinco proyectos actuales

**Metodologia:**
- Confirmar cuales variables tienen varianza cero dentro de cada proyecto
- Aplicar **Leave-One-Project-Out CV (LOGO-CV)**: entrenar en 4 proyectos,
  evaluar en el 5to; repetir 5 veces
- Comparar LOGO-CV con/sin variables de proyecto
- Construir variables relativas (asequibilidad) que capturen la relacion
  lead-proyecto de forma generalizable


In [ ]:
# === 7.8.1  Identificar variables de proyecto + crear variables relativas ===
PROYECTO_COL = 'proyecto'
assert PROYECTO_COL in df_raw.columns, f"Columna '{PROYECTO_COL}' no encontrada"

proyectos_unicos = sorted(df_raw[PROYECTO_COL].dropna().unique(), key=str)
print(f"Proyectos unicos ({len(proyectos_unicos)}): {proyectos_unicos}")
print()

CANDIDATE_PV = [
    'm2_minimo', 'm2_maximo',
    'precio_minimo', 'precio_maximo', 'precio_m2_promedio', 'precio_promedio',
    'zona_proyecto', 'subzona_proyecto', 'municipio_proyecto',
    PROYECTO_COL,
]
CANDIDATE_PV = [v for v in CANDIDATE_PV if v in df_raw.columns]

CONFIRMED_PROJECT_VARS = []
print('Analisis de varianza dentro de proyecto:')
for v in CANDIDATE_PV:
    per_proj = df_raw.groupby(PROYECTO_COL)[v].nunique(dropna=True)
    max_u = per_proj.max()
    if max_u <= 1:
        CONFIRMED_PROJECT_VARS.append(v)
        print(f'  [ID_PROYECTO] {v} — valor unico por proyecto (identificador)')
    else:
        print(f'  [VARIABLE   ] {v} — varia dentro de proyectos (max nunique={max_u})')

print(f'\nVariables que codifican identidad del proyecto ({len(CONFIRMED_PROJECT_VARS)}):')
print(f'  {CONFIRMED_PROJECT_VARS}')

print()
for fs_name, fs_list in FEATURE_SETS.items():
    proj_in_fs = [v for v in CONFIRMED_PROJECT_VARS if v in fs_list]
    if proj_in_fs:
        print(f'  {fs_name}: {len(proj_in_fs)} vars de proyecto -> {proj_in_fs}')

# ─── Convertir columnas numericas que pueden venir como string en el CSV ───
NUM_COLS_TO_CAST = [
    'm2_minimo', 'm2_maximo', 'precio_minimo', 'precio_maximo',
    'precio_promedio', 'precio_m2_promedio',
    'ingreso_familiar', 'ingreso', 'nucleo_familiar',
    'edad', 'participantes_en_credito', 'opciones_visitadas',
]
cast_report = []
for col in NUM_COLS_TO_CAST:
    if col not in df_raw.columns:
        continue
    if not pd.api.types.is_numeric_dtype(df_raw[col]):
        df_raw[col] = pd.to_numeric(df_raw[col], errors='coerce')
        cast_report.append(col)
if cast_report:
    print(f'\nColumnas convertidas a numerico (eran object/string): {cast_report}')
else:
    print('\nTodas las columnas numericas ya tienen dtype numerico.')

# ─── Crear variables relativas (generalizables a nuevos proyectos) ──────────
print()
print('Creando variables de asequibilidad...')
ENGINEERED_VARS = []

INGRESO_COL = next((c for c in ['ingreso_familiar', 'ingreso'] if c in df_raw.columns), None)

if INGRESO_COL and 'precio_promedio' in df_raw.columns:
    precio_ok  = df_raw['precio_promedio'].notna() & (df_raw['precio_promedio'] > 0)
    ingreso_ok = df_raw[INGRESO_COL].notna()
    df_raw['asequibilidad_indice'] = np.where(
        precio_ok & ingreso_ok,
        df_raw[INGRESO_COL] / df_raw['precio_promedio'], np.nan)
    ENGINEERED_VARS.append('asequibilidad_indice')
    pct = df_raw['asequibilidad_indice'].notna().mean() * 100
    med = df_raw['asequibilidad_indice'].median()
    print(f'  asequibilidad_indice = {INGRESO_COL}/precio_promedio  '
          f'({pct:.1f}% no nulos, mediana={med:.4f})')

if 'm2_minimo' in df_raw.columns and 'nucleo_familiar' in df_raw.columns:
    m2_ok     = df_raw['m2_minimo'].notna() & (df_raw['m2_minimo'] > 0)
    nucleo_ok = df_raw['nucleo_familiar'].notna()
    df_raw['m2_por_persona'] = np.where(
        m2_ok & nucleo_ok,
        df_raw['m2_minimo'] / df_raw['nucleo_familiar'].clip(lower=1), np.nan)
    ENGINEERED_VARS.append('m2_por_persona')
    pct = df_raw['m2_por_persona'].notna().mean() * 100
    med = df_raw['m2_por_persona'].median()
    print(f'  m2_por_persona = m2_minimo/nucleo_familiar  '
          f'({pct:.1f}% no nulos, mediana={med:.2f})')

if INGRESO_COL and 'm2_minimo' in df_raw.columns:
    m2_ok     = df_raw['m2_minimo'].notna() & (df_raw['m2_minimo'] > 0)
    ingreso_ok = df_raw[INGRESO_COL].notna()
    df_raw['ingreso_por_m2'] = np.where(
        ingreso_ok & m2_ok,
        df_raw[INGRESO_COL] / df_raw['m2_minimo'], np.nan)
    ENGINEERED_VARS.append('ingreso_por_m2')
    pct = df_raw['ingreso_por_m2'].notna().mean() * 100
    med = df_raw['ingreso_por_m2'].median()
    print(f'  ingreso_por_m2 = {INGRESO_COL}/m2_minimo  '
          f'({pct:.1f}% no nulos, mediana={med:.1f})')

if not ENGINEERED_VARS:
    print('  ADVERTENCIA: No se crearon variables de asequibilidad.')
    ingreso_cols = [c for c in df_raw.columns if 'ingreso' in c.lower()]
    print(f'  Columnas con "ingreso" en df_raw: {ingreso_cols}')

# ─── Definir F6 (lead-only) y F7 (asequibilidad) ─────────────────────────────
f4_key = next((k for k in FEATURE_SETS if k.upper().startswith('F4')), None)
F4_ALL_VARS = FEATURE_SETS.get(f4_key, []) if f4_key else []

F6_LEAD_ONLY     = [f for f in F4_ALL_VARS if f not in CONFIRMED_PROJECT_VARS]
F7_ASEQUIBILIDAD = F6_LEAD_ONLY + [v for v in ENGINEERED_VARS if v not in F6_LEAD_ONLY]

print()
print(f'F4_Todas:         {len(F4_ALL_VARS)} variables')
print(f'F6_Lead_Solo:     {len(F6_LEAD_ONLY)} variables  (F4 sin vars de proyecto)')
elim = [v for v in F4_ALL_VARS if v in CONFIRMED_PROJECT_VARS]
print(f'  Eliminadas de F4: {elim}')
print(f'F7_Asequibilidad: {len(F7_ASEQUIBILIDAD)} variables  (F6 + asequibilidad)')
if ENGINEERED_VARS:
    print(f'  Variables agregadas: {ENGINEERED_VARS}')
else:
    print('  NOTA: F7 == F6 (no se crearon vars de asequibilidad)')


In [ ]:
# === 7.8.2  Leave-One-Project-Out CV (LOGO-CV) ===
# Pregunta: cuanto cae el PR-AUC cuando el modelo NO conoce el proyecto de prueba?
# Se usa el dataset COMPLETO (diferente del split train/test principal, es una evaluacion adicional)
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import average_precision_score as aps

best_r_all = max(
    [r for r in all_results_conv if 'cluster' not in r.get('feature_set','').lower()],
    key=lambda r: r.get('pr_auc', 0)
)
alg_diag = best_r_all['algorithm']
fs_diag  = best_r_all['feature_set']
print(f'Algoritmo diagnostico: {alg_diag} (mejor PR-AUC hold-out: {best_r_all["pr_auc"]:.4f})')
print(f'Feature set original : {fs_diag}')
print()

logo_configs = {
    'Original (con vars proyecto)': FEATURE_SETS.get(fs_diag, []),
    'F6_Lead_Solo':                  F6_LEAD_ONLY,
    'F7_Asequibilidad':              F7_ASEQUIBILIDAD,
}

logo_scores = {k: [] for k in logo_configs}
grupos_all  = df_raw[PROYECTO_COL].values
y_all_arr   = np.asarray(y_conv)
logo_cv     = LeaveOneGroupOut()

for fold_i, (tr_logo, te_logo) in enumerate(
        logo_cv.split(df_raw, y_all_arr, grupos_all)):
    proy_test  = df_raw.iloc[te_logo][PROYECTO_COL].iloc[0]
    n_pos_test = int(y_all_arr[te_logo].sum())
    n_test     = len(te_logo)
    print(f'Fold {fold_i+1} — Proyecto test: {proy_test!r:25s}  n={n_test:,}  positivos={n_pos_test}')

    if n_pos_test < 5:
        print(f'  -> Muy pocos positivos, saltando este fold')
        for k in logo_configs:
            logo_scores[k].append(np.nan)
        continue

    for cfg_name, feat_list in logo_configs.items():
        avail_l = [f for f in feat_list if f in df_raw.columns]
        if len(avail_l) == 0:
            logo_scores[cfg_name].append(np.nan)
            continue
        cat_l, num_l = identify_col_types(df_raw, avail_l)
        use_ord_l    = (alg_diag != 'LR')
        est_l        = ALGORITHMS[alg_diag]()
        pipe_l       = build_pipeline(est_l, cat_l, num_l,
                                      use_smote=False, ordinal=use_ord_l)
        pipe_l.fit(df_raw.iloc[tr_logo][avail_l], y_all_arr[tr_logo])
        y_prob_l = pipe_l.predict_proba(df_raw.iloc[te_logo][avail_l])[:, 1]
        pr_l     = aps(y_all_arr[te_logo], y_prob_l)
        logo_scores[cfg_name].append(pr_l)
        print(f'  {cfg_name:35s} PR-AUC = {pr_l:.4f}')
    print()

print('=' * 65)
print('RESUMEN LOGO-CV')
print('=' * 65)
print(f"  {'Feature Set':35s}  {'Media':6s}  {'Std':5s}  {'vs hold-out':11s}")
print('  ' + '-'*62)
logo_summary = {}
for cfg_name, scores in logo_scores.items():
    valid  = [s for s in scores if not np.isnan(s)]
    mean_  = np.mean(valid) if valid else np.nan
    std_   = np.std(valid)  if valid else np.nan
    logo_summary[cfg_name] = mean_
    if cfg_name.startswith('Original'):
        holdout = best_r_all.get('pr_auc', np.nan)
    elif 'F6' in cfg_name:
        best_f6 = max((r for r in all_results_conv
                       if r.get('feature_set','').startswith('F6')),
                      key=lambda r: r.get('pr_auc', 0), default=None)
        holdout = best_f6['pr_auc'] if best_f6 else np.nan
    else:
        holdout = np.nan
    gap = mean_ - holdout if not np.isnan(holdout) and not np.isnan(mean_) else np.nan
    gap_str = f'{gap:+.4f}' if not np.isnan(gap) else '     N/A'
    print(f"  {cfg_name:35s}  {mean_:6.4f}  {std_:5.4f}  {gap_str}")

print()
orig_mean = logo_summary.get('Original (con vars proyecto)', np.nan)
f6_mean   = logo_summary.get('F6_Lead_Solo', np.nan)
f7_mean   = logo_summary.get('F7_Asequibilidad', np.nan)
if not np.isnan(orig_mean) and not np.isnan(f6_mean):
    caida_absoluta = orig_mean - f6_mean
    caida_rel      = caida_absoluta / orig_mean * 100 if orig_mean > 0 else 0
    print(f'  Caida al eliminar vars proyecto: {caida_absoluta:+.4f} ({caida_rel:.1f}%)')
    if not np.isnan(f7_mean):
        recuperacion = f7_mean - f6_mean
        print(f'  Recuperacion con asequibilidad  : {recuperacion:+.4f}')
    print()
    if caida_absoluta > 0.03:
        print('  DIAGNOSTICO: Las vars de proyecto contribuyen significativamente.')
        if not np.isnan(f7_mean) and f7_mean > f6_mean + 0.005:
            print('               F7_Asequibilidad recupera parte del desempeno.')
            print('               -> RECOMENDADO: usar F7 (generalizable y competitivo)')
        else:
            print('               Las variables relativas no recuperan el desempeno completamente.')
            print('               -> El modelo depende de especificidades de los 5 proyectos actuales.')
            print('               -> Documentar esta limitacion en la tesis.')
    else:
        print('  DIAGNOSTICO: Las vars de proyecto no son el driver principal.')
        print('               El modelo aprende patrones generalizables del lead.')


### 7.9 - Ingenieria de Caracteristicas para Generalizacion

Se construyen **variables relativas** que capturan la relacion lead-proyecto
sin codificar la identidad del proyecto:

| Variable | Formula | Interpretacion | Generalizable |
|----------|---------|----------------|---------------|
| `asequibilidad_indice` | ingreso / precio_promedio | Fraccion del precio que representa el ingreso | Si |
| `m2_por_persona` | m2_minimo / nucleo_familiar | Metros cuadrados disponibles por persona | Si |
| `ingreso_por_m2` | ingreso / m2_minimo | Poder adquisitivo por m2 de vivienda | Si |

**Por que generalizan:** un nuevo proyecto con precio diferente al de los 5
actuales mantiene el mismo significado del ratio. Un lead con
`asequibilidad_indice = 0.03` (ingreso = 3% del precio) se comporta igual
independientemente del precio absoluto del proyecto.

**Nuevos feature sets:**
- **F6_Lead_Solo**: Variables exclusivas del lead (sin ninguna variable de proyecto)
- **F7_Asequibilidad**: F6 + variables relativas lead x proyecto


In [ ]:
# === 7.9.1  Registrar feature sets y ejecutar experimentos de generalizacion ===
FEATURE_SETS['F6_Lead_Solo']     = F6_LEAD_ONLY
FEATURE_SETS['F7_Asequibilidad'] = F7_ASEQUIBILIDAD

print('Feature sets registrados:')
print(f'  F6_Lead_Solo:     {len(F6_LEAD_ONLY)} variables')
print(f'  F7_Asequibilidad: {len(F7_ASEQUIBILIDAD)} variables  (+{ENGINEERED_VARS})')
print()

gen_feature_sets = {'F6_Lead_Solo': F6_LEAD_ONLY, 'F7_Asequibilidad': F7_ASEQUIBILIDAD}

print('Entrenando modelos — Exp A (sin SMOTE)...')
results_conv_gen_a = run_experiment(
    df_raw, y_conv, idx_train_conv, idx_test_conv,
    gen_feature_sets, 'Conversion',
    use_smote=False, algorithms=ALGORITHMS
)

print('Entrenando modelos — Exp B (con SMOTE)...')
results_conv_gen_b = run_experiment(
    df_raw, y_conv, idx_train_conv, idx_test_conv,
    gen_feature_sets, 'Conversion',
    use_smote=True, smote_strategy=0.10, algorithms=ALGORITHMS
)

all_results_conv += results_conv_gen_a + results_conv_gen_b

df_results_conv = pd.DataFrame([
    {k: v for k, v in r.items()
     if k not in ('pipeline','y_prob_test','y_true_test','bm','pct_sel','cum_gain')}
    for r in all_results_conv
])
df_results_conv.to_csv(RESULTS_DIR / 'results_conv_all.csv', index=False)

# Show new results
df_gen = df_results_conv[df_results_conv['feature_set'].isin(gen_feature_sets.keys())].copy()
print()
print(f'Resultados F6 y F7 ({len(df_gen)} modelos):')
cols_s = ['algorithm','feature_set','experiment','pr_auc','roc_auc','lift_10pct','rec_10pct']
print(df_gen.sort_values('pr_auc', ascending=False)[cols_s].to_string(index=False))


In [ ]:
# === 7.9.2  Tabla comparativa: hold-out vs LOGO-CV por feature set ===
# Muestra el tradeoff: desempeno en proyectos conocidos vs generalizabilidad

# Build comparison: best hold-out PR-AUC per feature set vs LOGO-CV mean
fs_order = ['F1_Base','F2_Base+Demo','F3_Base+Comerc','F4_Todas',
            'F6_Lead_Solo','F7_Asequibilidad']

# Standardize logo config names for lookup
logo_lookup = {
    'Original (con vars proyecto)': logo_summary.get('Original (con vars proyecto)', np.nan),
    'F6_Lead_Solo'                : logo_summary.get('F6_Lead_Solo', np.nan),
    'F7_Asequibilidad'            : logo_summary.get('F7_Asequibilidad', np.nan),
}

print('COMPARACION HOLD-OUT vs LOGO-CV')
print('Hold-out = PR-AUC en test set con el mismo split de 5 proyectos')
print('LOGO-CV  = PR-AUC entrenando en 4 proyectos, prediciendo en el 5to')
print()
print(f"  {'Feature Set':20s}  {'Best Algo':5s}  {'Hold-out':8s}  {'LOGO-CV':8s}  {'Gap':7s}  Nota")
print('  ' + '-'*80)

for fs in fs_order:
    rows = df_results_conv[df_results_conv['feature_set'] == fs]
    if len(rows) == 0:
        continue
    best_row  = rows.loc[rows['pr_auc'].idxmax()]
    holdout   = float(best_row['pr_auc'])
    best_alg  = str(best_row['algorithm'])

    # Find LOGO-CV for this feature set
    if fs.startswith('F6'):
        logo_m = logo_lookup.get('F6_Lead_Solo', np.nan)
    elif fs.startswith('F7'):
        logo_m = logo_lookup.get('F7_Asequibilidad', np.nan)
    elif fs == fs_diag or fs.startswith(fs_diag.split('_')[0]):
        logo_m = logo_lookup.get('Original (con vars proyecto)', np.nan)
    else:
        logo_m = np.nan

    gap     = logo_m - holdout if not np.isnan(logo_m) else np.nan
    has_pv  = any(v in FEATURE_SETS.get(fs,[]) for v in CONFIRMED_PROJECT_VARS)
    nota    = 'vars proyecto' if has_pv else 'solo lead'
    if fs.startswith('F7'):
        nota = 'asequibilidad'

    logo_str = f'{logo_m:.4f}' if not np.isnan(logo_m) else '   N/A'
    gap_str  = f'{gap:+.4f}' if not np.isnan(gap) else '    N/A'
    print(f"  {fs:20s}  {best_alg:5s}  {holdout:8.4f}  {logo_str:8s}  {gap_str:7s}  {nota}")

print()
f6_holdout = df_results_conv[df_results_conv['feature_set']=='F6_Lead_Solo']['pr_auc'].max() if 'F6_Lead_Solo' in df_results_conv['feature_set'].values else np.nan
f7_holdout = df_results_conv[df_results_conv['feature_set']=='F7_Asequibilidad']['pr_auc'].max() if 'F7_Asequibilidad' in df_results_conv['feature_set'].values else np.nan
f4_holdout = df_results_conv[df_results_conv['feature_set']==f4_key]['pr_auc'].max() if f4_key and f4_key in df_results_conv['feature_set'].values else np.nan

print('INTERPRETACION:')
if not np.isnan(f7_holdout) and not np.isnan(f4_holdout):
    diff_f7_f4 = f7_holdout - f4_holdout
    print(f'  F7 vs F4 (hold-out): {diff_f7_f4:+.4f}')
    if diff_f7_f4 >= -0.01:
        print('  -> F7_Asequibilidad mantiene desempeno comparable a F4 sin codificar identidad del proyecto.')
        print('     RECOMENDADO para produccion: mayor capacidad de generalizacion.')
    elif diff_f7_f4 >= -0.03:
        print('  -> F7_Asequibilidad cae ligeramente vs F4. Considerar si la generalizabilidad')
        print('     justifica la caida segun el contexto de produccion.')
    else:
        print('  -> Caida importante vs F4. Documentar limitacion: el modelo actual depende')
        print('     de las caracteristicas especificas de los 5 proyectos disponibles.')
print(f'  Los feature sets F6 y F7 estan disponibles para la seleccion en Sec 7.10.')


### 7.10 - Seleccion de Modelos Candidatos

Antes de iniciar el refinamiento se identifican los **Top 5 modelos candidatos** ordenados por PR-AUC considerando **todos los experimentos** (A_NoSMOTE y B_SMOTE), sin limitar la busqueda al experimento A. Los candidatos seran refinados de forma independiente en las etapas siguientes.


In [ ]:
# === 7.10 Seleccion de modelos candidatos (todos los experimentos) ===
N_CANDIDATES = 5

top_candidates = (
    df_results_conv[
        ~df_results_conv['feature_set'].str.contains('cluster', case=False, na=False)
    ]
    .sort_values('pr_auc', ascending=False)
    .drop_duplicates(subset=['algorithm','feature_set','experiment'])
    .head(N_CANDIDATES)
    .reset_index(drop=True)
)

print('TOP CANDIDATOS - TODOS LOS EXPERIMENTOS (por PR-AUC)')
cols_show = ['algorithm','feature_set','experiment','pr_auc','roc_auc','recall','lift_10pct']
print(top_candidates[cols_show].to_string(index=True))

candidate_results_conv = []
for _, row in top_candidates.iterrows():
    match = next(
        (r for r in all_results_conv
         if r.get('algorithm') == row['algorithm']
         and r.get('feature_set') == row['feature_set']
         and r.get('experiment') == row['experiment']
         and 'pipeline' in r),
        None
    )
    if match:
        candidate_results_conv.append(match)
    else:
        print(f"  WARNING: no pipeline para {row['algorithm']} | {row['feature_set']} | {row['experiment']}")

print(f'\n{len(candidate_results_conv)} candidatos con pipeline disponible.')


### 7.11 - Etapa 1: Refinamiento de Variables

Para cada modelo candidato se computa la **importancia de permutacion**: se mide cuanto se degrada el PR-AUC cuando se aleatoriza una variable. Si una variable tiene importancia cero o negativa, no aporta informacion discriminativa.

**Criterio de eliminacion:** variables con importancia relativa **< 1% del total de importancia positiva acumulada**.

**Nota:** `HistGradientBoostingClassifier` no expone `feature_importances_`; se usa importancia de permutacion para todos los algoritmos, garantizando consistencia metodologica.


In [ ]:
# === 7.11.1 Funciones de refinamiento de variables ===
from sklearn.inspection import permutation_importance as perm_imp_fn
from sklearn.metrics import (average_precision_score, roc_auc_score,
                             f1_score, precision_score, recall_score)


def compute_perm_importance(result, n_repeats=5, sample_size=5000, seed=SEED):
    # Permutation importance using PR-AUC. Subsamples test set for speed.
    pipe       = result['pipeline']
    fs         = FEATURE_SETS.get(result['feature_set'], [])
    avail      = [f for f in fs if f in df_raw.columns]
    cat_cols, num_cols = identify_col_types(df_raw, avail)
    feat_names = cat_cols + num_cols

    X_test_raw = df_raw.iloc[idx_test_conv][avail]
    y_test     = np.array(y_conv[idx_test_conv])

    n   = min(sample_size, len(X_test_raw))
    rng = np.random.RandomState(seed)
    idx_s = rng.choice(len(X_test_raw), n, replace=False)
    X_s   = X_test_raw.iloc[idx_s]
    y_s   = y_test[idx_s]

    clf  = pipe.named_steps['clf']
    prep = pipe.named_steps['prep']
    X_t  = prep.transform(X_s)

    pi = perm_imp_fn(clf, X_t, y_s, n_repeats=n_repeats,
                     random_state=seed, scoring='average_precision')

    df_pi = pd.DataFrame({
        'feature'         : feat_names[:X_t.shape[1]],
        'importance_mean' : pi.importances_mean,
        'importance_std'  : pi.importances_std,
    }).sort_values('importance_mean', ascending=False).reset_index(drop=True)
    return df_pi, feat_names, avail


def get_features_to_keep(df_pi, threshold_rel=0.01):
    # Keep features with importance >= threshold_rel * total positive importance.
    pos_total = df_pi[df_pi['importance_mean'] > 0]['importance_mean'].sum()
    if pos_total <= 0:
        return list(df_pi['feature'])
    cutoff = threshold_rel * pos_total
    keep   = list(df_pi[df_pi['importance_mean'] >= cutoff]['feature'])
    return keep if len(keep) >= 3 else list(df_pi['feature'].iloc[:3])


def retrain_on_features(alg_name, feature_list, use_smote, task='conv'):
    # Rebuild and retrain same algorithm on a custom feature list.
    avail = [f for f in feature_list if f in df_raw.columns]
    if not avail:
        return None
    cat_c, num_c = identify_col_types(df_raw, avail)
    use_ord  = (alg_name != 'LR')
    est      = ALGORITHMS[alg_name]()
    pipe_new = build_pipeline(est, cat_c, num_c,
                              use_smote=use_smote, smote_strategy=0.10,
                              ordinal=use_ord)
    if task == 'conv':
        idx_tr, idx_te, y_all = idx_train_conv, idx_test_conv, y_conv
    else:
        idx_tr, idx_te, y_all = idx_train_aband, idx_test_aband, y_aband

    X_tr = df_raw.iloc[idx_tr][avail]
    X_te = df_raw.iloc[idx_te][avail]
    y_tr = y_all[idx_tr]
    y_te = np.array(y_all[idx_te])

    pipe_new.fit(X_tr, y_tr)
    y_prob = pipe_new.predict_proba(X_te)[:, 1]
    y_pred = (y_prob >= 0.5).astype(int)
    return {
        'pipeline'  : pipe_new,
        'features'  : avail,
        'n_features': len(avail),
        'pr_auc'    : average_precision_score(y_te, y_prob),
        'roc_auc'   : roc_auc_score(y_te, y_prob),
        'f1'        : f1_score(y_te, y_pred, zero_division=0),
        'recall'    : recall_score(y_te, y_pred, zero_division=0),
        'precision' : precision_score(y_te, y_pred, zero_division=0),
        'y_prob'    : y_prob,
        'y_true'    : y_te,
    }


print('Funciones definidas: compute_perm_importance, get_features_to_keep, retrain_on_features')


In [ ]:
# === 7.11.2 Analisis de importancia y refinamiento por candidato ===
PERM_REPEATS   = 5
PERM_SAMPLE    = 5000
DROP_THRESHOLD = 0.01

refinement_conv = []

for cand_idx, r in enumerate(candidate_results_conv):
    alg       = r['algorithm']
    fs        = r['feature_set']
    exp       = r['experiment']
    use_smote = (exp == 'B_SMOTE')

    print(f"\n{'-'*65}")
    print(f"Candidato {cand_idx+1}: {alg} | {fs} | {exp}")
    print(f"  PR-AUC base: {r['pr_auc']:.4f}  |  ROC-AUC: {r['roc_auc']:.4f}")

    df_pi, feat_names, avail_orig = compute_perm_importance(
        r, n_repeats=PERM_REPEATS, sample_size=PERM_SAMPLE)

    keep_feats = get_features_to_keep(df_pi, threshold_rel=DROP_THRESHOLD)
    drop_feats = [f for f in avail_orig if f not in keep_feats]
    print(f"  Originales={len(avail_orig)}  Eliminadas={len(drop_feats)}  Retenidas={len(keep_feats)}")
    if drop_feats:
        print(f"    Eliminadas: {drop_feats}")

    if drop_feats:
        ref = retrain_on_features(alg, keep_feats, use_smote, task='conv')
        if ref:
            d_pr  = ref['pr_auc'] - r['pr_auc']
            d_roc = ref['roc_auc'] - r['roc_auc']
            print(f"  PR-AUC refinado : {ref['pr_auc']:.4f}  (Delta={d_pr:+.4f})")
            print(f"  ROC-AUC refinado: {ref['roc_auc']:.4f}  (Delta={d_roc:+.4f})")
        else:
            ref = None
            print('  ERROR reentrenando - se mantiene conjunto original.')
    else:
        ref = None
        print('  Ninguna variable eliminada - modelo ya es minimo.')

    refinement_conv.append({
        'candidate_id'   : cand_idx,
        'algorithm'      : alg,
        'feature_set'    : fs,
        'experiment'     : exp,
        'use_smote'      : use_smote,
        'importance_df'  : df_pi,
        'features_orig'  : avail_orig,
        'features_keep'  : keep_feats,
        'features_drop'  : drop_feats,
        'pr_auc_orig'    : r['pr_auc'],
        'roc_auc_orig'   : r['roc_auc'],
        'pipeline_orig'  : r['pipeline'],
        'y_prob_orig'    : np.array(r.get('y_prob_test', [])),
        'refined'        : ref,
        'pr_auc_final'   : ref['pr_auc']   if ref else r['pr_auc'],
        'roc_auc_final'  : ref['roc_auc']  if ref else r['roc_auc'],
        'pipeline_final' : ref['pipeline'] if ref else r['pipeline'],
        'features_final' : ref['features'] if ref else avail_orig,
        'y_prob_final'   : ref['y_prob']   if ref else np.array(r.get('y_prob_test', [])),
        'y_true'         : np.array(y_conv[idx_test_conv]),
    })

print(f"\n{'='*65}")
print('  RESUMEN REFINAMIENTO - TAREA 1: CONVERSION')
print('='*65)
print(f"  {'#':>2}  {'Alg':>5}  {'FeatureSet':>16}  {'Exp':>10}  "
      f"{'N_orig':>6}  {'N_ref':>5}  {'PR_orig':>7}  {'PR_ref':>7}  {'Delta':>7}")
print('  ' + '-'*72)
for d in refinement_conv:
    n_r   = len(d['features_final'])
    pr_r  = d['pr_auc_final']
    delta = pr_r - d['pr_auc_orig']
    print(f"  {d['candidate_id']+1:>2}  {d['algorithm']:>5}  {d['feature_set']:>16}  "
          f"{d['experiment']:>10}  {len(d['features_orig']):>6}  {n_r:>5}  "
          f"{d['pr_auc_orig']:>7.4f}  {pr_r:>7.4f}  {delta:>+7.4f}")


In [ ]:
# === 7.11.3 Visualizacion: importancia de variables por candidato ===
from matplotlib.patches import Patch

n_cands = len(refinement_conv)
fig, axes = plt.subplots(1, n_cands, figsize=(4.5 * n_cands, 7))
if n_cands == 1:
    axes = [axes]

for ax, d in zip(axes, refinement_conv):
    df_pi  = d['importance_df'].head(15)
    colors = ['#EF5350' if f in d['features_drop'] else '#42A5F5'
              for f in df_pi['feature']]
    n = len(df_pi)
    ax.barh(range(n), df_pi['importance_mean'][::-1].values,
            xerr=df_pi['importance_std'][::-1].values,
            color=colors[::-1], edgecolor='white', alpha=0.9, capsize=3)
    ax.set_yticks(range(n))
    ax.set_yticklabels(df_pi['feature'][::-1].tolist(), fontsize=8)
    ax.axvline(0, color='black', lw=0.8)
    ax.set_title(f"{d['algorithm']} | {d['feature_set']}\n{d['experiment']}",
                 fontweight='bold', fontsize=9)
    ax.set_xlabel('Importancia permutacion (Delta PR-AUC)')

handles = [Patch(color='#42A5F5', label='Retenida'),
           Patch(color='#EF5350', label='Eliminada (< 1% importancia relativa)')]
fig.legend(handles=handles, loc='lower center', ncol=2, bbox_to_anchor=(0.5, -0.06))
plt.suptitle('Etapa 1 - Importancia de Permutacion por Candidato (Tarea 1: Conversion)',
             fontweight='bold', fontsize=12)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'refinamiento_variables_conv.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figura: figures/refinamiento_variables_conv.png')


### 7.12 - Etapa 2: Optimizacion del Umbral de Decision

El umbral de clasificacion convierte la probabilidad predicha en etiqueta binaria. El valor por defecto de **0.5 raramente es optimo** con datasets desbalanceados: con 1.18% de positivos, practicamente ningun lead lo superaria.

Se evaluan **umbrales de 0.05 a 0.95** calculando Precision, Recall, F1, Specificity, Accuracy, Lift, N leads priorizados y % del dataset.

**Umbrales de referencia:**
1. **F1-maximo** - maximiza media armonica precision-recall.
2. **Recall-maximo** (con Precision >= 10%) - mayor cobertura con precision minima.
3. **Operativo Top-10%** - umbral que prioriza aprox. el 10% del dataset.


In [ ]:
# === 7.12.1 Funciones de barrido de umbrales ===

def threshold_sweep(y_true, y_prob, thresholds=None):
    # Compute key classification metrics at each decision threshold.
    if thresholds is None:
        thresholds = np.round(np.arange(0.05, 0.96, 0.01), 2)
    y_true_arr = np.asarray(y_true, dtype=int)
    n_pos   = int(y_true_arr.sum())
    n_total = len(y_true_arr)
    base_rate = n_pos / n_total
    rows = []
    for thr in thresholds:
        y_pred = (y_prob >= thr).astype(int)
        tp = int(((y_pred==1) & (y_true_arr==1)).sum())
        fp = int(((y_pred==1) & (y_true_arr==0)).sum())
        tn = int(((y_pred==0) & (y_true_arr==0)).sum())
        fn = int(((y_pred==0) & (y_true_arr==1)).sum())
        prec = tp/(tp+fp) if (tp+fp) > 0 else 0.0
        rec  = tp/(tp+fn) if (tp+fn) > 0 else 0.0
        f1   = 2*prec*rec/(prec+rec) if (prec+rec) > 0 else 0.0
        spec = tn/(tn+fp) if (tn+fp) > 0 else 0.0
        acc  = (tp+tn) / n_total
        n_pp = int(y_pred.sum())
        pct  = n_pp / n_total * 100
        lift = prec / base_rate if base_rate > 0 and prec > 0 else 0.0
        rows.append({'threshold': thr, 'precision': prec, 'recall': rec,
                     'f1': f1, 'specificity': spec, 'accuracy': acc,
                     'lift': lift, 'n_positive': n_pp, 'pct_positive': pct,
                     'tp': tp, 'fp': fp, 'tn': tn, 'fn': fn})
    return pd.DataFrame(rows)


def find_optimal_thresholds(df_sw, min_precision=0.10, target_pct=10.0):
    # Return dict of optimal thresholds: f1_max, recall_max, operational.
    idx_f1 = df_sw['f1'].idxmax()
    thr_f1 = float(df_sw.loc[idx_f1, 'threshold'])

    ok = df_sw[df_sw['precision'] >= min_precision]
    if len(ok) > 0:
        idx_rc = ok['recall'].idxmax()
        thr_rc = float(ok.loc[idx_rc, 'threshold'])
        row_rc = ok.loc[idx_rc]
    else:
        thr_rc = thr_f1
        row_rc = df_sw.loc[idx_f1]

    df_sw2 = df_sw.copy()
    df_sw2['dist_op'] = (df_sw2['pct_positive'] - target_pct).abs()
    idx_op = df_sw2['dist_op'].idxmin()
    thr_op = float(df_sw2.loc[idx_op, 'threshold'])

    return {
        'f1_max'      : {'threshold': thr_f1, 'row': df_sw.loc[idx_f1]},
        'recall_max'  : {'threshold': thr_rc, 'row': row_rc},
        'operational' : {'threshold': thr_op, 'row': df_sw2.loc[idx_op]},
    }


print('Funciones definidas: threshold_sweep, find_optimal_thresholds')


In [ ]:
# === 7.12.2 Barrido de umbrales y visualizacion por candidato ===
n_cands = len(refinement_conv)
fig, axes = plt.subplots(n_cands, 3, figsize=(15, 4.5 * n_cands))
if n_cands == 1:
    axes = axes.reshape(1, -1)

threshold_summaries_conv = []

for row_i, d in enumerate(refinement_conv):
    y_prob = d['y_prob_final']
    y_true = d['y_true']
    if len(y_prob) == 0:
        print(f"Candidato {d['candidate_id']+1}: sin probabilidades, saltando.")
        continue

    df_sw = threshold_sweep(y_true, y_prob)
    opt   = find_optimal_thresholds(df_sw, min_precision=0.10, target_pct=10.0)
    d['threshold_df']  = df_sw
    d['threshold_opt'] = opt

    lbl = f"{d['algorithm']} | {d['feature_set']}"
    thr_colors = {'f1_max': '#42A5F5', 'recall_max': '#66BB6A', 'operational': '#FFA726'}
    ax0, ax1, ax2 = axes[row_i, 0], axes[row_i, 1], axes[row_i, 2]

    ax0.plot(df_sw['threshold'], df_sw['precision'], '#42A5F5', lw=1.8, label='Precision')
    ax0.plot(df_sw['threshold'], df_sw['recall'],    '#EF5350', lw=1.8, label='Recall')
    ax0.plot(df_sw['threshold'], df_sw['f1'],        '#66BB6A', lw=1.8, ls='--', label='F1')
    for k, p in opt.items():
        ax0.axvline(p['threshold'], color=thr_colors[k], linestyle=':', lw=1.3)
    ax0.set_title(f'{lbl}\nPrecision / Recall / F1 vs Threshold', fontsize=9, fontweight='bold')
    ax0.set_xlabel('Threshold'); ax0.set_ylabel('Metrica')
    ax0.legend(fontsize=8); ax0.grid(alpha=0.3)

    ax1.plot(df_sw['threshold'], df_sw['f1'], '#66BB6A', lw=2)
    ax1.axvline(opt['f1_max']['threshold'], color='#66BB6A', ls='--', lw=1.5,
                label=f"Max F1 @ {opt['f1_max']['threshold']:.2f}")
    ax1.set_title(f'{lbl}\nF1 vs Threshold', fontsize=9, fontweight='bold')
    ax1.set_xlabel('Threshold'); ax1.set_ylabel('F1')
    ax1.legend(fontsize=8); ax1.grid(alpha=0.3)

    ax2.plot(df_sw['threshold'], df_sw['pct_positive'], '#AB47BC', lw=2)
    ax2.axhline(10, color='#FFA726', ls='--', lw=1.5, label='Top 10% operativo')
    ax2.axvline(opt['operational']['threshold'], color='#FFA726', ls=':', lw=1.3)
    ax2.set_title(f'{lbl}\n% Leads priorizados vs Threshold', fontsize=9, fontweight='bold')
    ax2.set_xlabel('Threshold'); ax2.set_ylabel('% clasificados positivos')
    ax2.legend(fontsize=8); ax2.grid(alpha=0.3)

    row_f1 = opt['f1_max']['row']
    row_rc = opt['recall_max']['row']
    row_op = opt['operational']['row']
    threshold_summaries_conv.append({
        'candidate'         : f"{d['algorithm']}|{d['feature_set']}|{d['experiment']}",
        'thr_f1_max'        : opt['f1_max']['threshold'],
        'f1_at_thr'         : float(row_f1['f1']),
        'prec_at_f1'        : float(row_f1['precision']),
        'rec_at_f1'         : float(row_f1['recall']),
        'thr_recall_max'    : opt['recall_max']['threshold'],
        'rec_at_recall_max' : float(row_rc['recall']),
        'prec_at_recall_max': float(row_rc['precision']),
        'thr_op_10pct'      : opt['operational']['threshold'],
        'prec_at_op'        : float(row_op['precision']),
        'rec_at_op'         : float(row_op['recall']),
        'lift_at_op'        : float(row_op['lift']),
        'n_leads_op'        : int(row_op['n_positive']),
    })

plt.suptitle('Etapa 2 - Optimizacion del Umbral de Decision (Tarea 1: Conversion)',
             fontweight='bold', fontsize=12)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'threshold_optimization_conv.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figura: figures/threshold_optimization_conv.png')


In [ ]:
# === 7.12.3 Tabla resumen de umbrales optimos ===
df_thr_summary = pd.DataFrame(threshold_summaries_conv)

print('Threshold F1-maximo:')
print(df_thr_summary[['candidate','thr_f1_max','f1_at_thr','prec_at_f1','rec_at_f1']].to_string(index=False))
print()
print('Threshold Recall-maximo (Precision >= 10%):')
print(df_thr_summary[['candidate','thr_recall_max','rec_at_recall_max','prec_at_recall_max']].to_string(index=False))
print()
print('Threshold operativo Top-10%:')
print(df_thr_summary[['candidate','thr_op_10pct','prec_at_op','rec_at_op','lift_at_op','n_leads_op']].to_string(index=False))

for d, row in zip(refinement_conv, threshold_summaries_conv):
    d['thr_f1_max']   = row['thr_f1_max']
    d['thr_op_10pct'] = row['thr_op_10pct']

df_thr_summary.to_csv(RESULTS_DIR / 'threshold_optimization_conv.csv', index=False)
print('Guardado: results/threshold_optimization_conv.csv')


### 7.13 - Etapa 3: Seleccion Definitiva del Modelo

La seleccion considera **multiples criterios** ponderados:

| Criterio | Peso | Justificacion |
|----------|------|---------------|
| PR-AUC refinado | 30% | Metrica primaria en datasets desbalanceados |
| Lift@10% operativo | 25% | Valor practico para el equipo comercial |
| Recall@10% threshold | 20% | Cobertura de conversiones detectadas |
| ROC-AUC | 10% | Discriminacion general |
| Parsimonia (menos variables) | 10% | Interpretabilidad y mantenibilidad |
| Sin SMOTE | 5% | Preferencia por menor complejidad del pipeline |

El **umbral recomendado** es el threshold operativo Top-10%.


In [ ]:
# === 7.13 Seleccion definitiva - Tarea 1: Conversion ===

def composite_score_refined(d):
    # Multi-criteria composite score for refined candidate.
    pr_n  = d['pr_auc_final']
    roc_n = d['roc_auc_final']
    n_v   = len(d['features_final'])
    pars  = max(0, (20 - n_v) / 20)
    no_sm = 0.0 if d['use_smote'] else 1.0

    lift_n = 0.0
    rec_n  = 0.0
    if 'threshold_opt' in d and 'operational' in d['threshold_opt']:
        r_op   = d['threshold_opt']['operational']['row']
        lift_n = min(float(r_op['lift']), 15) / 15
        rec_n  = float(r_op['recall'])
    return (0.30 * pr_n + 0.25 * lift_n + 0.20 * rec_n +
            0.10 * roc_n + 0.10 * pars + 0.05 * no_sm)


for d in refinement_conv:
    d['composite_score'] = composite_score_refined(d)

ranked = sorted(refinement_conv, key=lambda d: d['composite_score'], reverse=True)

print('SELECCION DEFINITIVA - TAREA 1: CONVERSION')
print(f"  {'#':>2}  {'Alg':>5}  {'FeatureSet':>16}  {'Exp':>10}  "
      f"{'PR_ref':>7}  {'N_vars':>6}  {'Score':>7}")
print('  ' + '-'*60)
for rank, d in enumerate(ranked, 1):
    marker = ' <<< SELECCIONADO' if rank == 1 else ''
    print(f"  {rank:>2}  {d['algorithm']:>5}  {d['feature_set']:>16}  "
          f"{d['experiment']:>10}  {d['pr_auc_final']:>7.4f}  "
          f"{len(d['features_final']):>6}  {d['composite_score']:>7.4f}{marker}")

best_conv           = ranked[0]
best_conv_pipeline  = best_conv['pipeline_final']
best_conv_features  = best_conv['features_final']
best_conv_threshold = best_conv.get('thr_op_10pct', 0.5)
best_conv_exp       = best_conv['experiment']

print(f"\n  Modelo seleccionado:")
print(f"    Algoritmo  : {best_conv['algorithm']}")
print(f"    FeatureSet : {best_conv['feature_set']} (refinado)")
print(f"    Variables  : {best_conv_features}")
print(f"    Experimento: {best_conv_exp}")
print(f"    PR-AUC     : {best_conv['pr_auc_final']:.4f}")
print(f"    ROC-AUC    : {best_conv['roc_auc_final']:.4f}")
print(f"    Threshold  : {best_conv_threshold:.2f} (operativo Top-10%)")


### 7.14 - Etapa 4: Validacion Cruzada del Modelo Seleccionado

La validacion cruzada se realiza **unicamente sobre el modelo seleccionado** y con su **conjunto refinado de variables**. Si el modelo proviene de un experimento con SMOTE, el pipeline mantiene SMOTE dentro de cada fold para evitar *data leakage*.

Se usa **StratifiedKFold de 5 folds** sobre el conjunto de entrenamiento. Metrica primaria: PR-AUC. Se reportan tambien ROC-AUC y F1 con gap train/test.


In [ ]:
# === 7.14 Validacion cruzada del modelo seleccionado - Tarea 1 ===
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import make_scorer

print(f"Modelo: {best_conv['algorithm']} | {best_conv['feature_set']} | {best_conv_exp}")
print(f"Variables ({len(best_conv_features)}): {best_conv_features}")

avail_cv       = [f for f in best_conv_features if f in df_raw.columns]
cat_cv, num_cv = identify_col_types(df_raw, avail_cv)
est_cv         = ALGORITHMS[best_conv['algorithm']]()
use_ord_cv     = (best_conv['algorithm'] != 'LR')
pipe_cv        = build_pipeline(est_cv, cat_cv, num_cv,
                                use_smote=best_conv['use_smote'], smote_strategy=0.10,
                                ordinal=use_ord_cv)

X_train_cv = df_raw.iloc[idx_train_conv][avail_cv]
y_train_cv = y_conv[idx_train_conv]

cv_skf    = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
pr_scorer = make_scorer(average_precision_score, needs_proba=True)
scorers   = {'pr_auc': pr_scorer, 'roc_auc': 'roc_auc', 'f1': 'f1'}

cv_res = cross_validate(pipe_cv, X_train_cv, y_train_cv,
                        cv=cv_skf, scoring=scorers,
                        return_train_score=True, n_jobs=-1)

print()
print(f"  {'Metrica':>8}  {'Test mean':>9}  {'Test std':>8}  {'Train mean':>10}  {'CV%':>5}")
print('  ' + '-'*45)
for metric in ['pr_auc', 'roc_auc', 'f1']:
    ts     = cv_res[f'test_{metric}']
    tr     = cv_res[f'train_{metric}']
    cv_pct = ts.std() / ts.mean() * 100 if ts.mean() > 0 else 0
    print(f"  {metric:>8}  {ts.mean():>9.4f}  {ts.std():>8.4f}  {tr.mean():>10.4f}  {cv_pct:>4.1f}%")

gap = cv_res['train_pr_auc'].mean() - cv_res['test_pr_auc'].mean()
if gap < 0.05:
    verdict = 'Sin sobreajuste significativo'
elif gap < 0.15:
    verdict = 'Sobreajuste moderado - considerar regularizacion'
else:
    verdict = 'Sobreajuste elevado - revisar complejidad'
print(f"\n  Gap train-test PR-AUC: {gap:.4f} - {verdict}")
print(f"  PR-AUC hold-out (Etapa 1): {best_conv['pr_auc_final']:.4f}")
print(f"  PR-AUC CV-5fold (media)  : {cv_res['test_pr_auc'].mean():.4f}")


### 7.15 - Etapa 5: Optimizacion de Hiperparametros

Con el modelo definitivo y sus variables refinadas se aplica `RandomizedSearchCV` (15 iteraciones, 3-fold CV, scorer = PR-AUC). El tuning se realiza **unicamente sobre el modelo ya seleccionado**.

Se compara desempeno antes/despues. Si el modelo tuneado no mejora, se mantiene el original.


In [ ]:
# === 7.15 Hyperparameter tuning - modelo seleccionado, Tarea 1 ===
import time
from sklearn.model_selection import RandomizedSearchCV

alg_tune   = best_conv['algorithm']
avail_tune = [f for f in best_conv_features if f in df_raw.columns]
cat_tune, num_tune = identify_col_types(df_raw, avail_tune)
use_ord_tune = (alg_tune != 'LR')

PARAM_DISTS = {
    'GBM': {
        'clf__max_iter'         : [200, 400, 600],
        'clf__learning_rate'    : [0.01, 0.05, 0.10, 0.15],
        'clf__max_leaf_nodes'   : [31, 63, 127],
        'clf__min_samples_leaf' : [20, 50, 100],
        'clf__l2_regularization': [0.0, 0.1, 1.0],
    },
    'RF': {
        'clf__n_estimators'     : [60, 120, 200],
        'clf__max_depth'        : [None, 10, 20, 30],
        'clf__min_samples_split': [2, 5, 10],
        'clf__max_features'     : ['sqrt', 'log2', 0.5],
    },
    'EXT': {
        'clf__n_estimators'     : [60, 120, 200],
        'clf__max_depth'        : [None, 10, 20],
        'clf__min_samples_split': [2, 5, 10],
        'clf__max_features'     : ['sqrt', 'log2', 0.5],
    },
    'LR': {
        'clf__C'       : [0.01, 0.1, 1.0, 10.0],
        'clf__solver'  : ['lbfgs', 'saga'],
        'clf__max_iter': [300, 500, 1000],
    },
}

param_dist   = PARAM_DISTS.get(alg_tune, {})
X_train_tune = df_raw.iloc[idx_train_conv][avail_tune]
y_train_tune = y_conv[idx_train_conv]
X_test_tune  = df_raw.iloc[idx_test_conv][avail_tune]
y_test_tune  = np.array(y_conv[idx_test_conv])

if param_dist:
    est_tune  = ALGORITHMS[alg_tune]()
    pipe_tune = build_pipeline(est_tune, cat_tune, num_tune,
                               use_smote=best_conv['use_smote'], smote_strategy=0.10,
                               ordinal=use_ord_tune)
    pr_scorer_tune = make_scorer(average_precision_score, needs_proba=True)
    rscv = RandomizedSearchCV(pipe_tune, param_dist, n_iter=15, cv=3,
                              scoring=pr_scorer_tune, random_state=SEED,
                              n_jobs=-1, verbose=0)
    print(f'Buscando hiperparametros para {alg_tune} ({len(avail_tune)} variables)...')
    t0 = time.time()
    rscv.fit(X_train_tune, y_train_tune)
    print(f'  Tiempo: {time.time()-t0:.1f}s')
    print(f'  Mejores params: {rscv.best_params_}')
    print(f'  Mejor PR-AUC CV (3-fold): {rscv.best_score_:.4f}')

    y_prob_tuned = rscv.best_estimator_.predict_proba(X_test_tune)[:, 1]
    pr_tuned     = average_precision_score(y_test_tune, y_prob_tuned)
    roc_tuned    = roc_auc_score(y_test_tune, y_prob_tuned)

    print(f"\n  {'Metrica':>8}  {'Antes':>8}  {'Despues':>8}  {'Delta':>7}")
    print(f"  {'PR-AUC':>8}  {best_conv['pr_auc_final']:>8.4f}  {pr_tuned:>8.4f}  "
          f"{pr_tuned - best_conv['pr_auc_final']:>+7.4f}")
    print(f"  {'ROC-AUC':>8}  {best_conv['roc_auc_final']:>8.4f}  {roc_tuned:>8.4f}  "
          f"{roc_tuned - best_conv['roc_auc_final']:>+7.4f}")

    if pr_tuned >= best_conv['pr_auc_final'] - 0.001:
        best_conv_pipeline_final = rscv.best_estimator_
        best_conv_pr_final       = pr_tuned
        best_conv_roc_final      = roc_tuned
        best_conv_y_prob_final   = y_prob_tuned
        print(f'  Modelo tuneado adoptado (PR-AUC = {pr_tuned:.4f})')
    else:
        best_conv_pipeline_final = best_conv['pipeline_final']
        best_conv_pr_final       = best_conv['pr_auc_final']
        best_conv_roc_final      = best_conv['roc_auc_final']
        best_conv_y_prob_final   = best_conv['y_prob_final']
        print(f'  Tuning no mejoro - se mantiene pre-tuning (PR-AUC = {best_conv_pr_final:.4f})')
else:
    print(f'Sin espacio de hiperparametros para {alg_tune}.')
    best_conv_pipeline_final = best_conv['pipeline_final']
    best_conv_pr_final       = best_conv['pr_auc_final']
    best_conv_roc_final      = best_conv['roc_auc_final']
    best_conv_y_prob_final   = best_conv['y_prob_final']


### 7.16 - Evaluacion Final del Modelo de Conversion

Evaluacion completa sobre el conjunto de prueba usando el **umbral operativo Top-10%** identificado en la Etapa 2.


In [ ]:
# === 7.16 Evaluacion final - Tarea 1: Conversion ===
from sklearn.metrics import (precision_recall_curve, roc_curve,
                             ConfusionMatrixDisplay)

y_test_final = np.array(y_conv[idx_test_conv])
y_prob_final = best_conv_y_prob_final
thr_final    = best_conv_threshold
y_pred_final = (y_prob_final >= thr_final).astype(int)

tp = int(((y_pred_final==1)&(y_test_final==1)).sum())
fp = int(((y_pred_final==1)&(y_test_final==0)).sum())
tn = int(((y_pred_final==0)&(y_test_final==0)).sum())
fn = int(((y_pred_final==0)&(y_test_final==1)).sum())
prec = tp/(tp+fp) if (tp+fp) > 0 else 0
rec  = tp/(tp+fn) if (tp+fn) > 0 else 0
f1_s = 2*prec*rec/(prec+rec) if (prec+rec) > 0 else 0
base = y_test_final.mean()
lift = prec/base if base > 0 else 0
n_pp = int(y_pred_final.sum())

print('EVALUACION FINAL - TAREA 1: CONVERSION')
print(f"  Modelo     : {best_conv['algorithm']} | {best_conv['feature_set']} (refinado)")
print(f"  Variables  : {len(best_conv_features)}")
print(f"  Threshold  : {thr_final:.2f} (operativo Top-10%)")
print(f"  TP={tp:,}  FP={fp:,}  TN={tn:,}  FN={fn:,}")
print(f"  Precision={prec:.4f}  Recall={rec:.4f}  F1={f1_s:.4f}")
print(f"  Leads priorizados={n_pp:,} ({n_pp/len(y_test_final)*100:.1f}% del test)")
print(f"  Lift={lift:.2f}x  PR-AUC={best_conv_pr_final:.4f}  ROC-AUC={best_conv_roc_final:.4f}")

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

fpr_f, tpr_f, _ = roc_curve(y_test_final, y_prob_final)
axes[0].plot(fpr_f, tpr_f, '#42A5F5', lw=2, label=f'ROC-AUC={best_conv_roc_final:.3f}')
axes[0].plot([0,1],[0,1],'k--', lw=1)
axes[0].set_title('Curva ROC - Modelo Final (Conversion)', fontweight='bold')
axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR')
axes[0].legend(); axes[0].grid(alpha=0.3)

pr_c, rc_c, _ = precision_recall_curve(y_test_final, y_prob_final)
axes[1].plot(rc_c, pr_c, '#EF5350', lw=2, label=f'PR-AUC={best_conv_pr_final:.3f}')
axes[1].axhline(base, color='gray', ls='--', lw=1, label=f'Baseline={base:.3f}')
axes[1].scatter(rec, prec, s=100, color='#42A5F5', zorder=5,
                label=f'Thr={thr_final:.2f}')
axes[1].set_title('Curva Precision-Recall - Modelo Final (Conversion)', fontweight='bold')
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)

ConfusionMatrixDisplay.from_predictions(
    y_test_final, y_pred_final, ax=axes[2],
    display_labels=['No convierte','Convierte'],
    colorbar=False, cmap='Blues')
axes[2].set_title(f'Matriz de Confusion (Thr={thr_final:.2f})', fontweight='bold')

plt.suptitle('Evaluacion Final - Tarea 1: Conversion', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'evaluacion_final_conv.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figura: figures/evaluacion_final_conv.png')


---
> **Hallazgos - Seccion 7 (Tarea 1: Conversion):** El analisis iterativo en 5 etapas identifica el modelo definitivo a partir de los Top 5 candidatos de todos los experimentos. La importancia de permutacion revela que variables realmente discriminan; el conjunto refinado elimina variables de ruido sin sacrificar PR-AUC. La optimizacion del umbral confirma que el default de 0.5 es inoperativo en este dataset desbalanceado (84:1): el threshold operativo Top-10% entrega Lift > 6x con Recall ~60% de las conversiones. La validacion cruzada de 5 folds confirma la estabilidad del modelo seleccionado. El modelo definitivo queda disponible como `best_conv_pipeline_final` con threshold `best_conv_threshold`.


## Sección 8 — TAREA 2: Predicción de Abandono de Leads

### Contexto de Negocio
Un lead que entra al pipeline y abandona sin convertir representa un costo operativo sin retorno. Identificar tempranamente a los leads con alta propensión a abandonar permite intervenciones proactivas: refuerzos en el seguimiento, ajuste de oferta, o reasignación de recursos.

### Definición del Target
- **Dataset:** leads *en alcance del proyecto* (` == 0`), 166 219 registros
- **Target:** `target_abandoned = 1` si el lead no convirtió (era in-scope pero se perdió)
- **Desbalance:** ~100:1 (la mayoría de los leads in-scope que no convirtieron sí se consideran abandonados)

> Este conjunto es significativamente más pequeño que el dataset de conversión, lo que permite entrenamientos más rápidos y facilita SMOTE completo.


In [ ]:
print("=" * 65)
print("  EXPERIMENTO A — Sin SMOTE | Tarea 2: Abandono")
print("=" * 65)
results_aband_A = run_experiment(
    X_df        = df_raw,
    y_arr       = y_aband,
    idx_train   = idx_train_aband,
    idx_test    = idx_test_aband,
    feature_sets= FEATURE_SETS,
    task_name   = 'Abandono',
    use_smote   = False,
)
print(f"Completados: {len([r for r in results_aband_A if 'error' not in r])}")


In [ ]:
print("=" * 65)
print("  EXPERIMENTO B — Con SMOTE | Tarea 2: Abandono")
print("=" * 65)
results_aband_B = run_experiment(
    X_df        = df_raw,
    y_arr       = y_aband,
    idx_train   = idx_train_aband,
    idx_test    = idx_test_aband,
    feature_sets= FEATURE_SETS,
    task_name   = 'Abandono',
    use_smote   = True,
    smote_strategy = 0.20,   # higher ratio feasible on smaller dataset
)
print(f"Completados: {len([r for r in results_aband_B if 'error' not in r])}")


In [ ]:
all_results_aband = results_aband_A + results_aband_B
df_results_aband = pd.DataFrame([
    {k: v for k, v in r.items()
     if k not in ('pipeline','y_prob_test','y_true_test','bm','pct_sel','cum_gain')}
    for r in all_results_aband if 'error' not in r
])

df_display_aband = df_results_aband[display_cols].sort_values('pr_auc', ascending=False)
print("TABLA COMPARATIVA — TAREA 2: ABANDONO")
print(df_display_aband.to_string(index=False))

df_results_aband.drop(columns=['pipeline','y_prob_test','y_true_test','bm','pct_sel','cum_gain'],
                      errors='ignore').to_csv(
    RESULTS_DIR / 'task2_abandonment_all_experiments.csv', index=False
)
print("\n✓ Resultados guardados")


In [ ]:
# Visualizations for Task 2
best_per_alg_aband = (
    df_results_aband.sort_values('pr_auc', ascending=False)
    .groupby('algorithm').head(1)
    .reset_index(drop=True)
)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
ax_roc, ax_pr, ax_gain = axes

for idx_m, (_, row) in enumerate(best_per_alg_aband.iterrows()):
    match = next(
        (r for r in all_results_aband
         if r.get('algorithm')==row['algorithm']
         and r.get('experiment')==row['experiment']
         and r.get('feature_set')==row['feature_set']
         and 'error' not in r), None)
    if match is None:
        continue
    y_t = match['y_true_test']
    y_p = match['y_prob_test']
    col = ['#2196F3','#FF5722','#4CAF50','#9C27B0','#FF9800'][idx_m % 5]
    lbl = f"{row['algorithm']}"

    fpr, tpr, _ = roc_curve(y_t, y_p)
    ax_roc.plot(fpr, tpr, lw=1.8, color=col, label=f'{lbl} ({roc_auc_score(y_t,y_p):.3f})')

    pr_v, rc_v, _ = precision_recall_curve(y_t, y_p)
    ax_pr.plot(rc_v, pr_v, lw=1.8, color=col, label=f'{lbl} ({average_precision_score(y_t,y_p):.3f})')

    n = len(y_t)
    order = np.argsort(y_p)[::-1]
    cum_g = np.cumsum(y_t[order]) / y_t.sum()
    ax_gain.plot(np.arange(1,n+1)/n, cum_g, lw=1.8, color=col, label=lbl)

ax_roc.plot([0,1],[0,1],'k--',lw=0.8); ax_roc.set_title('ROC (Abandono)', fontweight='bold')
ax_roc.legend(fontsize=7); ax_roc.set_xlabel('FPR'); ax_roc.set_ylabel('TPR')

ax_pr.axhline(y_test_aband.mean(),linestyle='--',color='gray',lw=0.8)
ax_pr.set_title('Precision-Recall (Abandono)', fontweight='bold')
ax_pr.legend(fontsize=7); ax_pr.set_xlabel('Recall'); ax_pr.set_ylabel('Precision')

ax_gain.plot([0,1],[0,1],'k--',lw=0.8,label='Aleatorio')
ax_gain.set_title('Ganancia Acumulada (Abandono)', fontweight='bold')
ax_gain.legend(fontsize=7)

plt.suptitle('Evaluación Visual — Tarea 2: Abandono', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'task2_roc_pr_gain.png', dpi=150, bbox_inches='tight')
plt.show()


---
> **Hallazgos — Sección 8 (Tarea 2: Abandono):** El mejor modelo es **GBM | F4_Todas | Experimento A (Sin SMOTE)** con **ROC-AUC = 0.816** y **PR-AUC = 0.996**. El PR-AUC elevado refleja la tasa base de abandono (98.82%) más que la capacidad discriminativa del modelo — el PR-AUC de una clase mayoritaria siempre tiende a 1. La evaluación operativa (Sección 9) muestra que, al ordenar leads por probabilidad *ascendente* de abandono (equivalente a probabilidad descendente de conversión), el modelo captura conversiones con **Lift@10% = 6.07x**, prácticamente idéntico al modelo de conversión directa. Ambos modelos aprenden el mismo patrón subyacente desde perspectivas complementarias, lo que valida la consistencia del diseño experimental.


## Sección 9 — Evaluación Operativa: Escenarios de Priorización

La evaluación operativa traduce el desempeño técnico del modelo en valor práctico para el proceso comercial. Se comparan dos escenarios de gestión de leads bajo restricciones de recursos:

| Escenario | Descripción | Metodología |
|-----------|-------------|-------------|
| **1 — Modelo** | Los leads se ordenan por probabilidad predicha y se atiende el top K% | Aplicación directa del modelo sobre el conjunto de prueba |
| **2 — Aleatorio** | Se selecciona un número equivalente de leads al azar | Simulación Monte Carlo (10 000 iteraciones) |

Los umbrales evaluados son **5%, 10%, 20% y 30%** del conjunto de prueba.  
La comparación se realiza para **todos los algoritmos** (LR, RF, GBM, EXT) y **ambas tareas** (Conversión y Abandono), usando el mejor conjunto de características por algoritmo según PR-AUC.


### 9.1 Función de Evaluación Operativa

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# 9.1  Función de evaluación operativa (todos los modelos, ambas tareas)
# ════════════════════════════════════════════════════════════════════════

THRESHOLDS = [0.05, 0.10, 0.20, 0.30]
N_MC        = 10_000

def operational_evaluation(results_list, task_name,
                            thresholds=THRESHOLDS, n_mc=N_MC,
                            sort_ascending=False, seed=SEED):
    """
    Escenario 1: priorización por modelo para todos los resultados en results_list.
    Escenario 2: Monte Carlo sobre el mismo conjunto de prueba.

    sort_ascending=False → selecciona top K% por MAYOR probabilidad (conversión).
    sort_ascending=True  → selecciona top K% por MENOR probabilidad (abandono inverso).
    """
    rng = np.random.default_rng(seed)

    first_valid = next((r for r in results_list
                        if 'y_true_test' in r and 'y_prob_test' in r), None)
    if first_valid is None:
        print(f"  [WARN] No hay resultados válidos para {task_name}")
        return pd.DataFrame(), pd.DataFrame()

    y_true_ref  = first_valid['y_true_test']
    n_test      = len(y_true_ref)
    n_pos_total = int(y_true_ref.sum())
    base_rate   = n_pos_total / n_test

    # ── Escenario 2: Monte Carlo por umbral ──────────────────────────────
    mc_rows = []
    for thr in thresholds:
        n_sel = max(1, round(thr * n_test))
        mc_counts = np.array([
            int(y_true_ref[rng.choice(n_test, n_sel, replace=False)].sum())
            for _ in range(n_mc)
        ])
        mc_rows.append({
            'threshold_pct' : thr * 100,
            'n_selected'    : n_sel,
            'n_pos_total'   : n_pos_total,
            'base_rate_pct' : round(base_rate * 100, 4),
            'mc_mean'       : round(mc_counts.mean(), 2),
            'mc_std'        : round(mc_counts.std(), 2),
            'mc_ci95_lo'    : round(float(np.percentile(mc_counts,  2.5)), 2),
            'mc_ci95_hi'    : round(float(np.percentile(mc_counts, 97.5)), 2),
            'mc_precision'  : round(mc_counts.mean() / n_sel, 4),
            'mc_recall'     : round(mc_counts.mean() / n_pos_total, 4) if n_pos_total > 0 else 0,
        })
    df_mc = pd.DataFrame(mc_rows)

    # ── Escenario 1: modelo (todos los algoritmos × feature_sets) ────────
    model_rows = []
    for r in results_list:
        if 'y_prob_test' not in r or 'error' in r:
            continue
        y_prob = np.array(r['y_prob_test'])
        y_true = np.array(r['y_true_test'])
        n_t    = len(y_true)
        n_pos  = int(y_true.sum())

        if sort_ascending:
            order = np.argsort(y_prob)          # ascending (low prob first)
        else:
            order = np.argsort(y_prob)[::-1]    # descending (high prob first)

        y_sorted  = y_true[order]
        cum_pos   = np.cumsum(y_sorted)

        for thr in thresholds:
            n_sel       = max(1, round(thr * n_t))
            n_captured  = int(cum_pos[n_sel - 1])
            precision   = n_captured / n_sel
            recall      = n_captured / n_pos if n_pos > 0 else 0
            lift_base   = precision / base_rate if base_rate > 0 else 0

            mc_ref      = df_mc[df_mc['threshold_pct'] == thr * 100].iloc[0]
            lift_vs_mc  = n_captured / mc_ref['mc_mean'] if mc_ref['mc_mean'] > 0 else 0
            delta_conv  = n_captured - mc_ref['mc_mean']
            ci_lo, ci_hi = mc_ref['mc_ci95_lo'], mc_ref['mc_ci95_hi']
            sig_vs_mc   = n_captured > ci_hi  # above MC 97.5th percentile

            model_rows.append({
                'task'          : task_name,
                'algorithm'     : r.get('algorithm', ''),
                'feature_set'   : r.get('feature_set', ''),
                'experiment'    : r.get('experiment', ''),
                'threshold_pct' : thr * 100,
                'n_selected'    : n_sel,
                'n_converted'   : n_captured,
                'precision'     : round(precision, 4),
                'recall'        : round(recall, 4),
                'lift'          : round(lift_base, 4),
                'mc_mean'       : round(float(mc_ref['mc_mean']), 2),
                'mc_ci95_lo'    : ci_lo,
                'mc_ci95_hi'    : ci_hi,
                'lift_vs_mc'    : round(lift_vs_mc, 4),
                'delta_conv'    : round(delta_conv, 2),
                'sig_vs_mc'     : sig_vs_mc,
            })

    df_model = pd.DataFrame(model_rows)
    return df_model, df_mc

print("Función operational_evaluation definida.")
print(f"  Umbrales: {[f'{t*100:.0f}%' for t in THRESHOLDS]}")
print(f"  Iteraciones Monte Carlo: {N_MC:,}")


### 9.2 Escenario 1 — Priorización por Modelo (Todos los Algoritmos)

Para cada algoritmo se selecciona el mejor modelo (por PR-AUC) que predijo sobre el mismo conjunto de prueba (F1–F4). Se calcula el número de conversiones capturadas al contactar el top K% de leads ordenados por probabilidad predicha.


In [ ]:
# ════════════════════════════════════════════════════════════════════════
# 9.2  Escenario 1: Priorización por Modelo
# ════════════════════════════════════════════════════════════════════════

# ── Task 1: Conversión (F1–F4, mismo test set 81K leads) ─────────────────
results_conv_F1F4 = [r for r in all_results_conv
                     if r.get('feature_set') in ('F1_Base','F2_Base+Demo',
                                                    'F3_Base+Comercial','F4_Todas')]

print("Ejecutando evaluación operativa — Task 1: Conversión ...")
df_op_conv, df_mc_conv = operational_evaluation(
    results_conv_F1F4, task_name='Conversion',
    sort_ascending=False      # ordenar por mayor prob_conversion
)
print(f"  Modelos evaluados: {df_op_conv['algorithm'].nunique() if len(df_op_conv) else 0} algoritmos")

# ── Task 2: Abandono — objetivo: capturar leads que SÍ convirtieron ──────
# Para evaluar el valor operativo del modelo de abandono, se invierte la
# perspectiva: se busca capturar las conversiones (target_won=1 = y_aband=0)
# seleccionando primero los leads con MENOR prob_abandono.
# Esto equivale a ordenar descendentemente por (1 - prob_abandono) ≈ prob_conversion.
print("Ejecutando evaluación operativa — Task 2: Abandono (vista de conversión) ...")
results_aband_conv_view = []
for r in all_results_aband:
    if 'y_true_test' in r and 'y_prob_test' in r:
        r2 = {k: v for k, v in r.items()
               if k not in ('y_true_test','y_prob_test')}
        r2['y_true_test'] = 1 - np.array(r['y_true_test'])  # 1=convertido
        r2['y_prob_test']  = 1 - np.array(r['y_prob_test'])  # 1-prob_abandono
        results_aband_conv_view.append(r2)
    else:
        results_aband_conv_view.append(r)

df_op_aband, df_mc_aband = operational_evaluation(
    results_aband_conv_view,
    task_name='Abandono',
    sort_ascending=False   # mayor (1-prob_abandono) = menor riesgo de abandono
)
print(f"  Modelos evaluados: {df_op_aband['algorithm'].nunique() if len(df_op_aband) else 0} algoritmos")

# ── Tabla resumen por umbral — Task 1 ─────────────────────────────────────
print()
print("═" * 72)
print("  TABLA OPERATIVA — TAREA 1: CONVERSIÓN")
print("  (Conversiones capturadas en test set vs. selección aleatoria MC)")
print("═" * 72)
for thr_pct in [5, 10, 20, 30]:
    sub = df_op_conv[df_op_conv['threshold_pct'] == thr_pct]
    if len(sub) == 0:
        continue
    mc_val = df_mc_conv[df_mc_conv['threshold_pct'] == thr_pct]['mc_mean'].values[0]
    mc_ci  = df_mc_conv[df_mc_conv['threshold_pct'] == thr_pct][['mc_ci95_lo','mc_ci95_hi']].values[0]
    n_sel  = int(sub['n_selected'].iloc[0])
    n_pos  = int(sub['n_pos_total'].iloc[0]) if 'n_pos_total' in sub.columns else '-'
    print(f"\n--- Umbral {thr_pct:2d}% | n={n_sel:,} | Total conversiones={n_pos} | MC={mc_val:.1f} (IC95: {mc_ci[0]:.0f}–{mc_ci[1]:.0f}) ---")
    best_per = sub.loc[sub.groupby('algorithm')['n_converted'].idxmax()].copy()
    best_per = best_per.sort_values('n_converted', ascending=False)
    print(f"  {'Alg':5s}  {'Feat.Set':22s}  {'Conv':>6s}  {'Recall':>7s}  {'Prec':>7s}  {'Lift':>7s}  {'LvsMC':>7s}  Sig")
    for _, row in best_per.iterrows():
        sig = "***" if row['sig_vs_mc'] else "   "
        print(f"  {row['algorithm']:5s}  {row['feature_set']:22s}  {int(row['n_converted']):>6d}"
              f"  {row['recall']:>7.4f}  {row['precision']:>7.4f}  {row['lift']:>7.2f}x  {row['lift_vs_mc']:>7.2f}x  {sig}")

# ── Tabla resumen — Task 2 ─────────────────────────────────────────────
print()
print("═" * 72)
print("  TABLA OPERATIVA — TAREA 2: ABANDONO")
print("  Métrica: conversiones capturadas al priorizar leads de menor prob_abandono")
print("  (equivalente a ordenar por prob_conversion proxy = 1 - prob_abandono)")
print("═" * 72)
for thr_pct in [5, 10, 20, 30]:
    sub = df_op_aband[df_op_aband['threshold_pct'] == thr_pct]
    if len(sub) == 0:
        continue
    mc_val = df_mc_aband[df_mc_aband['threshold_pct'] == thr_pct]['mc_mean'].values[0]
    mc_ci  = df_mc_aband[df_mc_aband['threshold_pct'] == thr_pct][['mc_ci95_lo','mc_ci95_hi']].values[0]
    n_sel  = int(sub['n_selected'].iloc[0])
    print(f"\n--- Umbral {thr_pct:2d}% | n={n_sel:,} | MC={mc_val:.1f} (IC95: {mc_ci[0]:.0f}–{mc_ci[1]:.0f}) ---")
    best_per = sub.loc[sub.groupby('algorithm')['n_converted'].idxmax()].copy()
    best_per = best_per.sort_values('n_converted', ascending=False)
    print(f"  {'Alg':5s}  {'Feat.Set':22s}  {'Conv':>6s}  {'Recall':>7s}  {'Prec':>7s}  {'Lift':>7s}  {'LvsMC':>7s}  Sig")
    for _, row in best_per.iterrows():
        sig = "***" if row['sig_vs_mc'] else "   "
        print(f"  {row['algorithm']:5s}  {row['feature_set']:22s}  {int(row['n_converted']):>6d}"
              f"  {row['recall']:>7.4f}  {row['precision']:>7.4f}  {row['lift']:>7.2f}x  {row['lift_vs_mc']:>7.2f}x  {sig}")

# Save results
df_op_conv.to_csv(RESULTS_DIR / 'operational_eval_conversion.csv', index=False)
df_op_aband.to_csv(RESULTS_DIR / 'operational_eval_abandono.csv', index=False)
print(f"\nResultados guardados en results/")
print(f"  operational_eval_conversion.csv: {len(df_op_conv)} filas")
print(f"  operational_eval_abandono.csv   : {len(df_op_aband)} filas")


### 9.3 Escenario 2 — Línea Base Monte Carlo (10 000 iteraciones)

Para validar que las diferencias observadas no son producto del azar, se compara cada modelo con una distribución de referencia obtenida mediante 10 000 muestras aleatorias del mismo tamaño. Una diferencia estadísticamente significativa (por encima del IC 95%) se marca con `***`.


In [ ]:
# ════════════════════════════════════════════════════════════════════════
# 9.3  Línea Base Monte Carlo — Tabla Estadística Detallada
# ════════════════════════════════════════════════════════════════════════

print("══════════════════════════════════════════════════════════════")
print("  LÍNEA BASE MONTE CARLO — TAREA 1: CONVERSIÓN")
print("══════════════════════════════════════════════════════════════")
print(df_mc_conv.rename(columns={
    'threshold_pct': 'Umbral%', 'n_selected': 'N_sel', 'n_pos_total': 'N_pos_total',
    'base_rate_pct': 'BaseRate%', 'mc_mean': 'MC_media', 'mc_std': 'MC_std',
    'mc_ci95_lo': 'CI95_lo', 'mc_ci95_hi': 'CI95_hi',
    'mc_precision': 'MC_prec', 'mc_recall': 'MC_recall'
}).to_string(index=False))

print()
print("══════════════════════════════════════════════════════════════")
print("  LÍNEA BASE MONTE CARLO — TAREA 2: ABANDONO")
print("══════════════════════════════════════════════════════════════")
print(df_mc_aband.rename(columns={
    'threshold_pct': 'Umbral%', 'n_selected': 'N_sel', 'n_pos_total': 'N_pos_total',
    'base_rate_pct': 'BaseRate%', 'mc_mean': 'MC_media', 'mc_std': 'MC_std',
    'mc_ci95_lo': 'CI95_lo', 'mc_ci95_hi': 'CI95_hi',
    'mc_precision': 'MC_prec', 'mc_recall': 'MC_recall'
}).to_string(index=False))

print()
print("Nota: CI95 = intervalo de confianza del 95% para la selección aleatoria.")
print("Un modelo con n_converted > CI95_hi supera estadísticamente la aleatoriedad (marcado ***).")


### 9.4 Análisis Comparativo — Curvas de Ganancia Acumulada

Las curvas de ganancia acumulada muestran qué proporción de las conversiones totales se captura al contactar progresivamente el K% de leads mejor clasificados. Se incluye la banda del intervalo de confianza del 95% de la selección aleatoria (Monte Carlo).


In [ ]:
# ════════════════════════════════════════════════════════════════════════
# 9.4  Curvas de Ganancia Acumulada — Todos los Algoritmos
# ════════════════════════════════════════════════════════════════════════

ALGORITHMS_ORDER = ['LR', 'RF', 'GBM', 'EXT']
COLORS_ALG       = {'LR': '#5C6BC0', 'RF': '#43A047', 'GBM': '#E53935', 'EXT': '#FB8C00'}
N_MC_CURVE       = 2_000    # faster for the visual band

def compute_gain_curve(y_true, y_prob, sort_ascending=False):
    """Returns (pct_selected, pct_captured) arrays for a cumulative gain curve."""
    n  = len(y_true)
    nP = int(y_true.sum())
    if nP == 0:
        return np.linspace(0, 100, n), np.linspace(0, 100, n)
    if sort_ascending:
        order = np.argsort(y_prob)
    else:
        order = np.argsort(y_prob)[::-1]
    cum_pos     = np.cumsum(y_true[order])
    pct_sel     = np.arange(1, n + 1) / n * 100
    pct_cap     = cum_pos / nP * 100
    return pct_sel, pct_cap

def mc_gain_band(y_true, n_mc=N_MC_CURVE, seed=42):
    """Monte Carlo gain band at several thresholds."""
    rng = np.random.default_rng(seed)
    n   = len(y_true)
    nP  = int(y_true.sum())
    xs  = np.arange(1, 21) * 5   # 5%, 10%, ..., 100%
    mc_means, mc_lo, mc_hi = [], [], []
    for thr_pct in xs:
        n_sel = max(1, round(thr_pct / 100 * n))
        mc_c  = np.array([
            y_true[rng.choice(n, n_sel, replace=False)].sum()
            for _ in range(n_mc)
        ])
        mc_means.append(mc_c.mean() / nP * 100)
        mc_lo.append(np.percentile(mc_c,  2.5) / nP * 100)
        mc_hi.append(np.percentile(mc_c, 97.5) / nP * 100)
    return xs, np.array(mc_means), np.array(mc_lo), np.array(mc_hi)

# ── Figura 1: Task 1 — Conversión (2×2 por algoritmo) ────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Curvas de Ganancia Acumulada — Tarea 1: Conversión\n'
             '(Umbral K% → % conversiones capturadas)',
             fontsize=12, fontweight='bold')

for ax, alg in zip(axes.flat, ALGORITHMS_ORDER):
    alg_res = [r for r in results_conv_F1F4
               if r.get('algorithm') == alg and 'y_prob_test' in r]
    if not alg_res:
        ax.text(0.5, 0.5, f'{alg}\nsin resultados', ha='center', va='center')
        continue

    # Best result by PR-AUC
    best_r   = max(alg_res, key=lambda r: r.get('pr_auc', 0))
    y_true   = np.array(best_r['y_true_test'])
    y_prob   = np.array(best_r['y_prob_test'])
    n_pos    = int(y_true.sum())
    n_test_s = len(y_true)

    pct_sel, pct_cap = compute_gain_curve(y_true, y_prob, sort_ascending=False)
    xs_mc, mc_m, mc_lo, mc_hi = mc_gain_band(y_true)

    ax.plot(pct_sel, pct_cap,
            color=COLORS_ALG[alg], lw=2.5,
            label=f'{alg} ({best_r["feature_set"]})')
    ax.plot([0, 100], [0, 100], 'k--', lw=1.2, label='Aleatorio (media MC)')
    ax.fill_between(xs_mc, mc_lo, mc_hi, alpha=0.2, color='gray', label='IC 95% MC')

    # Perfect model line
    pct_ideal = n_pos / n_test_s * 100
    ax.plot([0, pct_ideal, 100], [0, 100, 100], 'g:', lw=1.2, alpha=0.7, label='Modelo perfecto')

    # Mark Lift@5, 10, 20, 30
    for thr_pct, marker in [(5,'o'),(10,'s'),(20,'^')]:
        n_sel_pt = max(1, round(thr_pct/100 * n_test_s))
        captured = int(np.cumsum(y_true[np.argsort(y_prob)[::-1]])[n_sel_pt-1])
        pct_c    = captured / n_pos * 100
        mc_ref   = df_mc_conv[df_mc_conv['threshold_pct'] == thr_pct]['mc_mean'].values[0]
        lift_mc  = captured / mc_ref if mc_ref > 0 else 0
        ax.annotate(f'{thr_pct}%: {lift_mc:.1f}x MC',
                    xy=(thr_pct, pct_c), xytext=(thr_pct+3, pct_c-5),
                    fontsize=7, color=COLORS_ALG[alg],
                    arrowprops=dict(arrowstyle='->', color=COLORS_ALG[alg], lw=0.8))

    ax.set_xlim(0, 100); ax.set_ylim(0, 105)
    ax.set_xlabel('% Leads Contactados')
    ax.set_ylabel('% Conversiones Capturadas')
    ax.set_title(f'Algoritmo: {alg}', fontweight='bold')
    ax.legend(fontsize=7, loc='lower right')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'operational_gain_curves_conv.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figura guardada: operational_gain_curves_conv.png")

# ── Figura 2: Task 2 — Abandono (2×2) ────────────────────────────────────
fig2, axes2 = plt.subplots(2, 2, figsize=(14, 10))
fig2.suptitle('Curvas de Ganancia Acumulada — Tarea 2: Abandono\n'
              '(Ordenado por prob_abandono ASCENDENTE = priorizar menor riesgo de abandono)',
              fontsize=12, fontweight='bold')

for ax, alg in zip(axes2.flat, ALGORITHMS_ORDER):
    alg_res = [r for r in results_aband_conv_view
               if r.get('algorithm') == alg and 'y_prob_test' in r]
    if not alg_res:
        ax.text(0.5, 0.5, f'{alg}\nsin resultados', ha='center', va='center')
        continue

    best_r   = max(alg_res, key=lambda r: r.get('roc_auc', 0))
    y_aband  = np.array(best_r['y_true_test'])    # 1=abandoned, 0=converted
    y_prob_a = np.array(best_r['y_prob_test'])    # prob_abandono
    # Objective: find leads NOT likely to abandon (y_aband=0)
    y_conv_proxy = 1 - y_aband                    # 1=converted
    n_pos_c      = int(y_conv_proxy.sum())
    n_test_s     = len(y_conv_proxy)

    if n_pos_c == 0:
        ax.text(0.5, 0.5, 'Sin conversiones en test', ha='center', va='center')
        continue

    # Sort ASCENDING by prob_abandono → capture low-risk leads first
    pct_sel, pct_cap = compute_gain_curve(y_conv_proxy, y_prob_a, sort_ascending=True)
    xs_mc, mc_m, mc_lo, mc_hi = mc_gain_band(y_conv_proxy)

    ax.plot(pct_sel, pct_cap, color=COLORS_ALG[alg], lw=2.5,
            label=f'{alg} ({best_r["feature_set"]})')
    ax.plot([0, 100], [0, 100], 'k--', lw=1.2, label='Aleatorio (media MC)')
    ax.fill_between(xs_mc, mc_lo, mc_hi, alpha=0.2, color='gray', label='IC 95% MC')
    pct_ideal_a = n_pos_c / n_test_s * 100
    ax.plot([0, pct_ideal_a, 100], [0, 100, 100], 'g:', lw=1.2, alpha=0.7, label='Modelo perfecto')

    ax.set_xlim(0, 100); ax.set_ylim(0, 105)
    ax.set_xlabel('% Leads Contactados (prob. abandono ↑)')
    ax.set_ylabel('% Conversiones Capturadas')
    ax.set_title(f'Algoritmo: {alg}', fontweight='bold')
    ax.legend(fontsize=7, loc='lower right')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'operational_gain_curves_aband.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figura guardada: operational_gain_curves_aband.png")

# ── Figura 3: Comparación de Lift@K por algoritmo (barras) ────────────────
fig3, axes3 = plt.subplots(1, 2, figsize=(14, 5))
fig3.suptitle('Lift vs. Selección Aleatoria (Monte Carlo) por Algoritmo',
              fontsize=12, fontweight='bold')

for task_ax, (df_op, task_lbl, df_mc_r) in zip(
        axes3,
        [(df_op_conv, 'Conversión', df_mc_conv),
         (df_op_aband, 'Abandono (inverso)', df_mc_aband)]):
    if len(df_op) == 0:
        task_ax.text(0.5, 0.5, 'Sin datos', ha='center', va='center')
        task_ax.set_title(f'Task: {task_lbl}')
        continue

    thr_show = 10   # Lift@10%
    sub      = df_op[df_op['threshold_pct'] == thr_show]
    best_per = sub.loc[sub.groupby('algorithm')['lift_vs_mc'].idxmax()]
    best_per = best_per.sort_values('lift_vs_mc', ascending=False)

    colors_b = [COLORS_ALG.get(a, '#90A4AE') for a in best_per['algorithm']]
    bars_b   = task_ax.bar(best_per['algorithm'], best_per['lift_vs_mc'],
                           color=colors_b, edgecolor='white', linewidth=0.5)
    task_ax.axhline(1.0, color='red', linestyle='--', lw=1.5, label='Aleatorio = 1.0x')

    for bar_b, row_b in zip(bars_b, best_per.itertuples()):
        task_ax.text(bar_b.get_x() + bar_b.get_width()/2,
                     bar_b.get_height() + 0.02,
                     f'{row_b.lift_vs_mc:.2f}x',
                     ha='center', va='bottom', fontsize=9, fontweight='bold')

    task_ax.set_xlabel('Algoritmo')
    task_ax.set_ylabel('Lift vs. Monte Carlo Aleatorio')
    task_ax.set_title(f'Lift@{thr_show}% — Task: {task_lbl}', fontweight='bold')
    task_ax.legend(fontsize=8)
    task_ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'operational_lift_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Figura guardada: operational_lift_comparison.png")


---
> **Hallazgos — Sección 9 (Evaluación Operativa):** La priorización basada en modelos supera significativamente la selección aleatoria en todos los umbrales. Al contactar el **5% de los leads mejor clasificados** (~4,080 de 81,606 del set de prueba), el modelo captura el **41.5% de todas las conversiones** (Lift@5% = **8.3x**). Al **10%**, el Lift = **6.05x** (583 conversiones capturadas, Recall = 60.5%). La simulación Monte Carlo (10,000 iteraciones) confirma que todos los resultados son estadísticamente significativos (p < 0.025, marcados ***). Los modelos de árbol (GBM, EXT, RF) logran lifts indistinguibles entre sí; la Regresión Logística obtiene Lift@5% = 6.77x, competitiva pero inferior. El conjunto de características **F2_Base+Demo es el más eficiente** operativamente: máximo rendimiento con mínima complejidad de recolección de datos.


## Sección 10 — Interpretación de Modelos

### 10.1 — Coeficientes de Regresión Logística

La Regresión Logística ofrece interpretabilidad directa a través de sus coeficientes. Valores positivos aumentan la probabilidad de conversión; valores negativos la disminuyen.


In [ ]:
# Get LR pipeline from results
lr_match = next(
    (r for r in all_results_conv
     if r.get('algorithm') == 'LR'
     and r.get('experiment') == 'A_NoSMOTE'
     and 'error' not in r), None
)

if lr_match:
    pipe_lr = lr_match['pipeline']
    fs_lr   = FEATURE_SETS.get(lr_match['feature_set'], [])
    avail_lr = [f for f in fs_lr if f in df_raw.columns]
    cat_lr, num_lr = identify_col_types(df_raw, avail_lr)

    # Extract feature names after OHE
    prep_lr = pipe_lr.named_steps['prep']
    try:
        ohe = prep_lr.named_transformers_['cat'].named_steps['enc']
        ohe_names = list(ohe.get_feature_names_out(cat_lr))
    except:
        ohe_names = cat_lr
    feature_names_lr = ohe_names + num_lr

    lr_clf = pipe_lr.named_steps['clf']
    coefs  = lr_clf.coef_[0]

    df_coef = pd.DataFrame({
        'feature': feature_names_lr[:len(coefs)],
        'coef'   : coefs
    }).sort_values('coef', key=abs, ascending=False).head(20)

    fig, ax = plt.subplots(figsize=(10, 7))
    colors_coef = ['#EF5350' if c > 0 else '#42A5F5' for c in df_coef['coef']]
    ax.barh(range(len(df_coef)), df_coef['coef'], color=colors_coef, edgecolor='white')
    ax.set_yticks(range(len(df_coef)))
    ax.set_yticklabels(df_coef['feature'], fontsize=8)
    ax.axvline(0, color='black', lw=0.8)
    ax.set_title('Top 20 Coeficientes — Regresión Logística\n(Tarea 1: Conversión)',
                 fontweight='bold')
    ax.set_xlabel('Coeficiente (escala log-odds)')
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'lr_coefficients_conv.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("\nTop 10 variables positivas (aumentan probabilidad de conversión):")
    print(df_coef[df_coef['coef']>0].head(10)[['feature','coef']].to_string(index=False))
    print("\nTop 10 variables negativas (reducen probabilidad de conversión):")
    print(df_coef[df_coef['coef']<0].head(10)[['feature','coef']].to_string(index=False))
else:
    print("Modelo LR no disponible para interpretación de coeficientes")


### 10.2 — Feature Importance (Tree-Based Models)

Los modelos basados en árboles (RF, GBM, XGB, LGB) proveen importancia de características basada en la reducción de impureza (Gini importance). Se visualizan las 20 variables más importantes del mejor modelo.


In [ ]:
# ════════════════════════════════════════════════════════════════════════
# 10.2  Feature Importance — Best Tree Model with feature_importances_
# (HistGradientBoosting has no feature_importances_; prefer RF/EXT which do)
# ════════════════════════════════════════════════════════════════════════

# Prefer RF or EXT (they have feature_importances_), then GBM as fallback
best_tree_match = max(
    (r for r in all_results_conv
     if r.get('algorithm') in ('RF', 'EXT')
     and r.get('experiment') == 'A_NoSMOTE'
     and 'error' not in r),
    key=lambda r: r.get('pr_auc', 0),
    default=None
)
if best_tree_match is None:
    # Fallback to any tree model
    best_tree_match = max(
        (r for r in all_results_conv
         if r.get('algorithm') in ('GBM', 'RF', 'EXT')
         and r.get('experiment') == 'A_NoSMOTE'
         and 'error' not in r),
        key=lambda r: r.get('pr_auc', 0),
        default=None
    )

if best_tree_match:
    alg_n = best_tree_match['algorithm']
    fs_t  = FEATURE_SETS.get(best_tree_match['feature_set'], [])
    avail_t = [f for f in fs_t if f in df_raw.columns]
    cat_t, num_t = identify_col_types(df_raw, avail_t)

    pipe_tree = best_tree_match['pipeline']
    clf_tree  = pipe_tree.named_steps['clf']
    feat_names_tree = cat_t + num_t

    importances = getattr(clf_tree, 'feature_importances_', None)
    if importances is not None and len(importances) == len(feat_names_tree):
        df_imp = pd.DataFrame({
            'feature'    : feat_names_tree,
            'importance' : importances
        }).sort_values('importance', ascending=False).head(20)

        fig, ax = plt.subplots(figsize=(10, 7))
        ax.barh(range(len(df_imp)), df_imp['importance'],
                color='#5C6BC0', edgecolor='white', alpha=0.9)
        ax.set_yticks(range(len(df_imp)))
        ax.set_yticklabels(df_imp['feature'], fontsize=8)
        ax.invert_yaxis()
        ax.set_title(f'Top 20 Variables Más Importantes — {alg_n}\n(Tarea 1: Conversión, {best_tree_match["feature_set"]})',
                     fontweight='bold')
        ax.set_xlabel('Importancia (Gini)')
        plt.tight_layout()
        plt.savefig(FIGURES_DIR / f'feature_importance_{alg_n}_conv.png', dpi=150, bbox_inches='tight')
        plt.show()
        print(f"Modelo: {alg_n} | {best_tree_match['feature_set']} | PR-AUC={best_tree_match['pr_auc']:.4f}")
        print("\nTop 20 variables más importantes:")
        print(df_imp.to_string(index=False))
    else:
        print(f"{alg_n} no expone feature_importances_ (HistGradientBoosting no lo soporta).")
        print("Usando importancia de permutación no es factible aquí; consultar SHAP (10.3).")
        df_imp = None
else:
    print("No hay modelo de árbol disponible para importancia de características.")
    df_imp = None


### 10.3 — Análisis SHAP

SHAP (SHapley Additive exPlanations) cuantifica la contribución de cada variable a la predicción de un caso individual, con garantías de consistencia y eficiencia derivadas de la teoría de juegos cooperativos.

Se computan valores SHAP sobre una muestra representativa del test set para el mejor modelo.


In [ ]:
# ════════════════════════════════════════════════════════════════════════
# 10.3  SHAP Analysis — best tree model on Conversion task
# ════════════════════════════════════════════════════════════════════════
# Try best tree model in order of preference
# Select best tree model by PR-AUC for SHAP interpretation
shap_match = max(
    (r for r in all_results_conv
     if r.get('algorithm') in ('GBM', 'RF', 'EXT')
     and r.get('experiment') == 'A_NoSMOTE'
     and 'error' not in r
     and 'y_prob_test' in r),
    key=lambda r: r.get('pr_auc', 0),
    default=None
)

if shap_match:
    fs_shap    = FEATURE_SETS.get(shap_match['feature_set'], F1_BASE)
    avail_shap = [f for f in fs_shap if f in df_raw.columns]
    cat_shap, num_shap = identify_col_types(df_raw, avail_shap)

    pipe_shap = shap_match['pipeline']
    clf_shap  = pipe_shap.named_steps['clf']
    prep_shap = pipe_shap.named_steps['prep']

    X_test_raw_shap = df_raw.iloc[idx_test_conv][avail_shap]
    SHAP_SAMPLE     = min(2000, len(X_test_raw_shap))
    sample_idx_shap = np.random.choice(len(X_test_raw_shap), SHAP_SAMPLE, replace=False)
    X_test_sample   = X_test_raw_shap.iloc[sample_idx_shap]

    X_test_transformed = prep_shap.transform(X_test_sample)
    feat_names_shap    = cat_shap + num_shap

    print(f"Modelo SHAP: {shap_match['algorithm']} | {shap_match['feature_set']}")
    print(f"Muestras para SHAP: {SHAP_SAMPLE}, features transformadas: {X_test_transformed.shape[1]}")

    try:
        explainer   = shap.TreeExplainer(clf_shap)
        shap_raw    = explainer(X_test_transformed)

        # shap_raw can be an Explanation object (shap >= 0.40) or a numpy array
        if hasattr(shap_raw, 'values'):
            sv = shap_raw.values
        else:
            sv = shap_raw

        # Binary classification may give shape (n, p, 2) or (2, n, p) or (n, p)
        if sv.ndim == 3:
            if sv.shape[-1] == 2:
                sv = sv[:, :, 1]          # positive class, shape (n, p)
            elif sv.shape[0] == 2:
                sv = sv[1]                # shape (n, p)
        elif isinstance(sv, list):
            sv = sv[1] if len(sv) == 2 else sv[0]

        # Trim feature names to match
        n_feats = sv.shape[1]
        feat_names_plot = feat_names_shap[:n_feats]

        print(f"SHAP values shape: {sv.shape}")

        # Summary plot (dot)
        fig, ax = plt.subplots(figsize=(10, 8))
        shap.summary_plot(sv, X_test_transformed[:, :n_feats],
                          feature_names=feat_names_plot,
                          max_display=20, show=False)
        plt.title(f"SHAP Summary — {shap_match['algorithm']} (Conversion)",
                  fontweight='bold', fontsize=12)
        plt.tight_layout()
        plt.savefig(FIGURES_DIR / 'shap_summary_conv.png', dpi=150, bbox_inches='tight')
        plt.show()

        # Bar plot (mean |SHAP|)
        fig, ax = plt.subplots(figsize=(10, 7))
        shap.summary_plot(sv, X_test_transformed[:, :n_feats],
                          feature_names=feat_names_plot,
                          plot_type='bar', max_display=20, show=False)
        plt.title(f"SHAP Mean |Value| — {shap_match['algorithm']} (Conversion)",
                  fontweight='bold', fontsize=12)
        plt.tight_layout()
        plt.savefig(FIGURES_DIR / 'shap_bar_conv.png', dpi=150, bbox_inches='tight')
        plt.show()
        print("SHAP analysis completed via TreeExplainer")

    except Exception as shap_err:
        print(f"TreeExplainer failed: {shap_err}")
        print("Falling back to KernelExplainer on small subsample (slow but robust)...")
        try:
            SHAP_SMALL = 100
            X_bg   = X_test_transformed[:50]
            X_eval = X_test_transformed[:SHAP_SMALL]
            ke = shap.KernelExplainer(
                lambda x: clf_shap.predict_proba(x)[:, 1], X_bg
            )
            sv_k = ke.shap_values(X_eval, nsamples=30, silent=True)
            if isinstance(sv_k, list):
                sv_k = sv_k[1]

            fig, ax = plt.subplots(figsize=(10, 7))
            shap.summary_plot(sv_k, X_eval,
                              feature_names=feat_names_shap[:X_eval.shape[1]],
                              plot_type='bar', max_display=15, show=False)
            plt.title("SHAP Mean |Value| — KernelExplainer (subsample 100)",
                      fontweight='bold')
            plt.tight_layout()
            plt.savefig(FIGURES_DIR / 'shap_bar_conv_kernel.png', dpi=150, bbox_inches='tight')
            plt.show()
            print("SHAP analysis completed via KernelExplainer fallback")
        except Exception as ke_err:
            print(f"KernelExplainer also failed: {ke_err}")
            print("SHAP analysis skipped — feature importance plot used instead")
else:
    print("No tree model available for SHAP analysis")


### 10.4 — Síntesis de Factores Determinantes

Se integran los tres métodos de interpretabilidad (coeficientes de regresión logística, importancia de características en árboles, y valores SHAP) en un ranking unificado. Las importancias se normalizan al rango [0, 1] antes de promediarlas, permitiendo comparar variables independientemente de la escala de cada método. Este análisis identifica los **factores que el equipo comercial debe priorizar** al evaluar la calidad de un lead.


In [ ]:
# ════════════════════════════════════════════════════════════════════════
# 10.4  Síntesis de Factores Determinantes — Ranking Unificado
# ════════════════════════════════════════════════════════════════════════
import numpy as np, pandas as pd

importance_sources = {}

# Source 1: SHAP mean absolute values
try:
    sv = np.array(shap_raw.values) if hasattr(shap_raw, 'values') else np.array(shap_raw)
    if sv.ndim == 3:
        sv = sv[:, :, 1]
    n_f = sv.shape[1]
    names_shap = feat_names_plot[:n_f] if 'feat_names_plot' in dir() else feat_names_shap[:n_f]
    shap_mean = pd.Series(np.abs(sv).mean(axis=0), index=names_shap, name='SHAP')
    importance_sources['SHAP'] = shap_mean
    print(f"✓ SHAP: {len(shap_mean)} variables  (modelo: {shap_match['algorithm']} | {shap_match['feature_set']})")
except Exception as e:
    print(f"  SHAP no disponible: {e}")

# Source 2: Tree feature importance (df_imp from cell 10.2)
try:
    imp_tree = df_imp.set_index('feature')['importance'].rename('Árbol')
    importance_sources['Árbol'] = imp_tree
    print(f"✓ Árbol: {len(imp_tree)} variables  (modelo: {alg_n} | {best_tree_match['feature_set']})")
except Exception as e:
    print(f"  Tree importance no disponible: {e}")

# Source 3: LR coefficient magnitudes (df_coef from cell 10.1)
try:
    imp_lr = df_coef.set_index('feature')['coef'].abs().rename('RegLog')
    importance_sources['RegLog'] = imp_lr
    print(f"✓ RegLog: {len(imp_lr)} variables")
except Exception as e:
    print(f"  LR coefs no disponible: {e}")

if not importance_sources:
    print("No hay fuentes de importancia disponibles.")
else:
    # Normalize each source to [0, 1]
    normalized = {}
    for name, series in importance_sources.items():
        mn, mx = series.min(), series.max()
        normalized[name] = (series - mn) / (mx - mn) if mx > mn else series * 0.0

    df_synth = pd.DataFrame(normalized).fillna(0)
    df_synth['ranking_promedio'] = df_synth.mean(axis=1)
    df_synth = df_synth.sort_values('ranking_promedio', ascending=False)

    print()
    print("╔══════════════════════════════════════════════════════════════════════╗")
    print("║        TOP 20 FACTORES DETERMINANTES — RANKING UNIFICADO            ║")
    print("╚══════════════════════════════════════════════════════════════════════╝")
    df_disp = df_synth.head(20).round(3).reset_index()
    df_disp.columns = df_disp.columns.map(str)
    df_disp.rename(columns={'index': 'Variable'}, inplace=True)
    df_disp.index = range(1, len(df_disp) + 1)
    print(df_disp.to_string())

    df_synth.reset_index().rename(columns={'index': 'variable'}).to_csv(
        RESULTS_DIR / 'factores_determinantes.csv', index=False
    )
    print(f"\n✓ Guardado: results/factores_determinantes.csv")

    # Plot
    top15 = df_synth.head(15)
    fig, ax = plt.subplots(figsize=(13, 6))
    x = np.arange(len(top15))
    width = 0.25
    palette = {'SHAP': '#42A5F5', 'Árbol': '#66BB6A', 'RegLog': '#FFA726'}
    offsets = {'SHAP': -width, 'Árbol': 0, 'RegLog': width}
    for method in importance_sources:
        col = top15.get(method, pd.Series(dtype=float))
        ax.bar(x + offsets.get(method, 0), col.values, width,
               label=method, color=palette.get(method, '#999'), alpha=0.85, edgecolor='white')
    ax.set_xticks(x)
    ax.set_xticklabels(top15.index.tolist(), rotation=45, ha='right', fontsize=9)
    ax.set_ylabel('Importancia normalizada [0–1]')
    ax.set_title('Top 15 Factores Determinantes — Importancia por Método de Interpretabilidad',
                 fontweight='bold', fontsize=12)
    ax.legend(title='Método')
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'factores_determinantes.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("✓ Figura: figures/factores_determinantes.png")


---
> **Hallazgos — Sección 10 (Interpretación de Modelos):** Los tres métodos de interpretabilidad convergen en las mismas variables como más determinantes. Las variables de **canal de adquisición** (`canal_de_contacto`, `canal_de_anuencia`, `fuente_pt2`) lideran consistentemente el ranking en SHAP y modelos de árbol: el canal por el que un lead entra al pipeline es el predictor más poderoso de conversión. Las variables de **proyecto y precio** (`proyecto`, `precio_lista_mxn`, `zona_proyecto`) ocupan los siguientes puestos, indicando que las características del inmueble discriminan significativamente la probabilidad de conversión. Las variables demográficas (`edad`, `sexo`, `estado_civil`) tienen importancia moderada y alta ausencia de datos. La Regresión Logística resalta el peso diferencial del canal digital vs. presencial; los modelos de árbol capturan interacciones no lineales entre canal, proyecto y precio que la LR no puede modelar, lo que explica su superioridad en PR-AUC.


## Sección 11 — Tabla Comparativa Global y Selección del Modelo Final

### 11.1 — Tabla Consolidada de Todos los Experimentos


In [ ]:
# Combine all results
all_experiments = (
    [r for r in all_results_conv  if 'error' not in r] +
    [r for r in all_results_aband if 'error' not in r]
)

df_global = pd.DataFrame([
    {k: v for k, v in r.items()
     if k not in ('pipeline','y_prob_test','y_true_test','bm','pct_sel','cum_gain')}
    for r in all_experiments
])

df_global.to_csv(RESULTS_DIR / 'all_experiments_comparison.csv', index=False)
print(f"Tabla global: {len(df_global)} experimentos")
print(f"  Conversión : {len(df_global[df_global['task']=='Conversion'])}")
print(f"  Abandono   : {len(df_global[df_global['task']=='Abandono'])}")
print()
print("TABLA GLOBAL — Ordenada por PR-AUC Descendente:")
print(df_global[display_cols + ['task']].sort_values('pr_auc', ascending=False).to_string(index=False))


### 11.2 — Criterios de Selección del Modelo Final

La selección no se basa en una sola métrica. Se ponderan múltiples criterios:

| Criterio | Peso | Justificación |
|----------|------|---------------|
| PR-AUC | Alto | Más informativo que ROC-AUC para datasets desbalanceados |
| Recall@10% | Alto | Operativamente crítico: captura de conversiones con recursos limitados |
| Lift@10% | Alto | Indica la ganancia real respecto a selección aleatoria |
| Generalización (gap train-CV) | Medio | Garantiza robustez en producción |
| Interpretabilidad | Medio | Importante para la defensa ante el comité académico |
| Tiempo de entrenamiento | Bajo | Considera viabilidad operativa |


In [ ]:
# Tabla comparativa global y seleccion definitiva — Sec 11
df_conv_final  = df_global[df_global['task'] == 'Conversion'].copy()
df_aband_final = df_global[df_global['task'] == 'Abandono'].copy()

# --- Abandono: seleccion por score compuesto (Tarea 2 no tiene metodologia iterativa) ---
def score_model(row):
    return (
        0.30 * row.get('pr_auc', 0) +
        0.25 * row.get('rec_10pct', 0) +
        0.25 * (row.get('lift_10pct', 0) / 10.0) +
        0.10 * row.get('roc_auc', 0) +
        0.10 * row.get('f1', 0)
    )

df_aband_final['composite_score'] = df_aband_final.apply(score_model, axis=1)
best_aband_model = df_aband_final.sort_values('composite_score', ascending=False).iloc[0]

# --- Conversion: modelo ya seleccionado en Sec 7 (metodologia iterativa) ---
print('MODELO DE CONVERSION (seleccionado en Sec 7 - metodologia iterativa):')
print(f"  Algoritmo  : {best_conv['algorithm']}")
print(f"  FeatureSet : {best_conv['feature_set']} (refinado - {len(best_conv_features)} variables)")
print(f"  Experimento: {best_conv_exp}")
print(f"  PR-AUC     : {best_conv_pr_final:.4f}")
print(f"  ROC-AUC    : {best_conv_roc_final:.4f}")
print(f"  Threshold  : {best_conv_threshold:.2f}")

print()
print('MODELO DE ABANDONO (seleccionado por score compuesto global):')
for k in ['algorithm','experiment','feature_set','pr_auc','roc_auc','lift_10pct']:
    print(f"  {k:20s}: {best_aband_model.get(k, 'N/A')}")


---
> **Hallazgos — Sección 11 (Comparación Global y Selección de Modelo):** Mediante un puntaje compuesto ponderando PR-AUC (30%), Recall@10% (25%), Lift@10% (25%), ROC-AUC (10%) y F1 (10%), el modelo seleccionado para **Tarea 1** es **GBM | F2_Base+Demo | A_NoSMOTE** (PR-AUC = 0.121, Lift@10% = 6.05x) y para **Tarea 2** es **GBM | F4_Todas | A_NoSMOTE** (ROC-AUC = 0.816). En ambas tareas el algoritmo **GBM (HistGradientBoostingClassifier)** domina, ofreciendo el mejor balance entre desempeño predictivo y eficiencia computacional (~0.5s/modelo en 408K filas). El conjunto **F2_Base+Demo** es suficiente para la conversión sin incurrir en el costo de recolección de todas las variables, validando que los datos de canal y fuente de adquisición son los de mayor valor informativo para el negocio.


## Sección 12 — Serialización de Modelos y Dataset de Scoring para Power BI

Se serializan los modelos finales para reutilización y se genera un dataset de scoring completo con las probabilidades predichas para cada lead, listo para importar en Power BI.


In [ ]:
# Guardar modelo de conversion (metodologia iterativa Sec 7)
import pickle
with open(MODELS_DIR / 'best_conversion_model.pkl', 'wb') as f:
    pickle.dump(best_conv_pipeline_final, f)
print(f"Modelo de conversion guardado: {best_conv['algorithm']} | "
      f"{best_conv['feature_set']} (refinado)")
print(f"  Variables ({len(best_conv_features)}): {best_conv_features}")
print(f"  Threshold: {best_conv_threshold:.2f}")

# Guardar modelo de abandono (Sec 11 - mejor global)
best_aband_match = next(
    (r for r in all_results_aband
     if r.get('algorithm') == best_aband_model['algorithm']
     and r.get('experiment') == best_aband_model['experiment']
     and r.get('feature_set') == best_aband_model['feature_set']
     and 'error' not in r), None
)
if best_aband_match:
    with open(MODELS_DIR / 'best_abandonment_model.pkl', 'wb') as f:
        pickle.dump(best_aband_match['pipeline'], f)
    print(f"Modelo de abandono guardado: {best_aband_model['algorithm']} | "
          f"{best_aband_model['feature_set']}")


In [ ]:
# Dataset de scoring (408K leads) usando modelo final de Sec 7
available_scoring = [f for f in best_conv_features if f in df_raw.columns]
prob_conv = best_conv_pipeline_final.predict_proba(df_raw[available_scoring])[:, 1]
pred_conv = (prob_conv >= best_conv_threshold).astype(int)

df_scoring = df_raw[['id_trato','proyecto','canal_de_anuencia',
                      'canal_de_contacto','fuente_pt2','zona_proyecto',
                      'target_won']].copy()
df_scoring['prob_conversion']    = prob_conv
df_scoring['pred_conversion']    = pred_conv
df_scoring['priority_score_pct'] = (
    df_scoring['prob_conversion'].rank(pct=True) * 100
).round(1)
df_scoring['priority_tier'] = pd.cut(
    df_scoring['priority_score_pct'],
    bins=[0, 80, 90, 95, 100],
    labels=['Bajo','Medio','Alto','Muy Alto']
)

if best_aband_match:
    fs_aband = FEATURE_SETS.get(best_aband_model['feature_set'], [])
    avail_aband = [f for f in fs_aband if f in df_raw.columns]
    prob_aband = best_aband_match['pipeline'].predict_proba(df_raw[avail_aband])[:, 1]
    df_scoring['prob_abandono'] = prob_aband

df_scoring.to_csv(RESULTS_DIR / 'scoring_dataset_powerbi.csv', index=False)
print(f"Dataset de scoring generado: {len(df_scoring):,} leads")
print(f"Columnas: {df_scoring.columns.tolist()}")
print()
print('Nulos en columnas de probabilidad:')
print(df_scoring[['prob_conversion','prob_abandono']].isnull().sum())
print()
print('Distribucion Tier de Prioridad:')
print(df_scoring['priority_tier'].value_counts())


## Sección 13 — Conclusiones y Recomendaciones

### 13.1 — Hallazgos Principales

#### Sobre el Desbalance de Clases
El dataset presenta un desbalance extremo (83:1 para conversión). Los modelos entrenados sin corrección de desbalance (Experimento A) tendieron a optimizar para la clase mayoritaria, resultando en alta accuracy pero bajo recall. El uso de `class_weight='balanced'` en combinación con métricas de evaluación adecuadas (PR-AUC, Lift@K) fue fundamental para identificar modelos con verdadera utilidad operativa.

#### Sobre los Conjuntos de Características
Las características de **canal de contacto y fuente de adquisición** resultaron las más predictivas para la conversión. El enriquecimiento con variables demográficas aportó mejoras modestas debido al alto porcentaje de datos faltantes (>50% en varias variables). Las variables comerciales (campaña, tipo de unidad) ofrecieron ganancia marginal sobre la base.

#### Sobre el Impacto de SMOTE
SMOTE (Experimento B) mejoró el recall a costa de menor precision en la mayoría de los modelos. Para el caso de uso de priorización de leads, un mayor recall es preferible (es mejor contactar un lead falso positivo que perder un verdadero positivo). Sin embargo, el umbral de decisión debe ajustarse según la capacidad operativa del equipo.

#### Sobre los Clústeres K-Means (F5)
La incorporación de la asignación de clúster como característica mostró **mejora marginal** en la mayoría de los modelos, confirmando que el K-Means capturó información comportamental relevante. Sin embargo, su disponibilidad limitada (42 K de los 408 K leads) restringe la aplicabilidad operativa.

#### Sobre la Interpretación
Las variables con mayor contribución SHAP son consistentes con la intuición de negocio: leads provenientes de **referencia directa y visita inédita** tienen mayor probabilidad de conversión que leads de campañas digitales masivas. El **proyecto** al que está interesado el lead también discrimina significativamente.

### 13.2 — Recomendación Final

Se recomienda desplegar el modelo de **GBM (HistGradientBoostingClassifier) sobre el conjunto de características F2 (Base+Demográficas)** para la predicción de conversión, por las siguientes razones:

1. **Rendimiento:** Mejor PR-AUC y Lift@10% entre todos los experimentos
2. **Velocidad:** Entrenamiento en ~0.5s sobre 408K filas, comparable a RF/EXT
3. **Generalización:** Gap train-test menor al 5% en ROC-AUC
4. **Practicidad:** Manejo nativo de valores faltantes; no requiere escalado
5. **Mantenibilidad:** Parámetros bien documentados, activa comunidad

### 13.3 — Recomendaciones Operativas

| Recomendación | Detalle |
|---------------|---------|
| **Umbral de decisión** | Ajustar a 0.3 (en lugar de 0.5) para maximizar recall en entornos de alta capacidad; 0.5 para entornos con restricción de recursos |
| **Retrenamiento** | Mensual o al acumular 5 000 leads nuevos |
| **Monitoreo** | Monitorear drift del Lift@10% en producción; alerta si cae más del 20% |
| **Enriquecimiento de datos** | Priorizar la captura de variables demográficas (edad, ingreso) que actualmente tienen >80% de nulos |
| **Integración Power BI** | Utilizar el archivo `scoring_dataset_powerbi.csv` generado; el campo `priority_tier` es un segmento operativo directo |

### 13.4 — Limitaciones y Trabajo Futuro

- **Temporalidad:** El modelo es estático; no captura tendencias temporales ni estacionalidad. Se recomienda explorar modelos con ventanas temporales deslizantes.
- **Variables faltantes:** El alto porcentaje de nulos en demografía reduce el poder predictivo. Un modelo específico para leads con datos completos podría superar al modelo general.
- **Causalidad:** Los modelos identifican correlaciones, no causas. La interpretación de coeficientes debe hacerse con cautela; las correlaciones pueden reflejar sesgos del proceso de ventas (ej. asesores contactan más a ciertos segmentos).
- **Validación prospectiva:** Se recomienda un experimento A/B en producción para validar el Lift real del modelo antes del despliegue masivo.


In [ ]:
# Final summary table
print("╔══════════════════════════════════════════════════════════════════════╗")
print("║                RESUMEN FINAL DEL ESTUDIO                            ║")
print("╚══════════════════════════════════════════════════════════════════════╝")
print(f"  Algoritmos evaluados        : 4 (LR, RF, GBM, EXT)")
print(f"  Conjuntos de características: 5 (F1–F5)")
print(f"  Experimentos por tarea      : 2 (A sin SMOTE, B con SMOTE)")
print(f"  Total de modelos entrenados : {len(df_global)}")
print(f"  Tareas modeladas            : 2 (Conversión, Abandono)")
print()
print(f"  Mejor modelo - Conversión:")
print(f"    {best_conv_model['algorithm']} | {best_conv_model['feature_set']} | {best_conv_model['experiment']}")
print(f"    PR-AUC={best_conv_model['pr_auc']:.4f}  ROC-AUC={best_conv_model['roc_auc']:.4f}  "
      f"Lift@10%={best_conv_model['lift_10pct']:.2f}x")
print()
print(f"  Mejor modelo - Abandono:")
print(f"    {best_aband_model['algorithm']} | {best_aband_model['feature_set']} | {best_aband_model['experiment']}")
print(f"    PR-AUC={best_aband_model['pr_auc']:.4f}  ROC-AUC={best_aband_model['roc_auc']:.4f}  "
      f"Lift@10%={best_aband_model['lift_10pct']:.2f}x")
print()
print("Archivos generados:")
for f in sorted(RESULTS_DIR.glob('*')) + sorted(MODELS_DIR.glob('*')):
    print(f"  {f.relative_to(BASE_DIR)}")
